# Redrob Hackathon Sandbox Notebook

This notebook is the required sandbox reproducibility check (Section 10.5 of the spec).

It runs the full ranking pipeline end to end on a small pre loaded sample of candidates,
using the same scoring logic as rank.py in the GitHub repo.

No file upload needed. No API key needed. No GPU required for the ranking step.


In [1]:
%pip install -q scikit-learn sentence-transformers


In [2]:
# Step 2: shared scoring logic
# This is the same code as common.py in the GitHub repo, copied here so the
# notebook is fully self contained and does not depend on external files.

"""
Shared logic for the Redrob candidate ranker.
Used by both precompute_embeddings.py (offline) and rank.py (timed step).
"""
import json
import gzip
import re
from datetime import date, datetime
# JD-derived constants (encoded from job_description.docx)


JD_TEXT = """
Senior AI Engineer, Founding Team, Redrob AI. Series A AI-native talent
intelligence platform. Own the intelligence layer: ranking, retrieval, and
matching systems for candidate-JD matching. Production experience with
embeddings-based retrieval systems (sentence-transformers, OpenAI embeddings,
BGE, E5) deployed to real users, handling embedding drift, index refresh,
retrieval-quality regression in production. Production experience with vector
databases or hybrid search infrastructure: Pinecone, Weaviate, Qdrant, Milvus,
OpenSearch, Elasticsearch, FAISS. Strong Python and code quality. Hands-on
experience designing evaluation frameworks for ranking systems: NDCG, MRR,
MAP, offline-to-online correlation, A/B testing. LLM fine-tuning (LoRA, QLoRA,
PEFT), learning-to-rank models, HR-tech or marketplace product background,
distributed systems, large-scale inference optimization, open-source
contributions are a plus. Ideal candidate: 6-8 years total experience, 4-5
years in applied ML/AI roles at product companies, has shipped an end-to-end
ranking, search, or recommendation system to real users at meaningful scale,
strong opinions on hybrid retrieval, offline vs online evaluation, and when to
fine-tune vs prompt an LLM, defensible with reference to systems actually
built.
"""

CONSULTING_FIRMS = {
    "tcs", "tata consultancy services", "infosys", "wipro", "accenture",
    "cognizant", "capgemini",
}

PREFERRED_LOCATIONS = {"pune", "noida"}
ACCEPTABLE_LOCATIONS = {"hyderabad", "mumbai", "delhi", "new delhi", "gurgaon", "gurugram", "delhi ncr"}

RETRIEVAL_KEYWORDS = [
    "embedding", "retrieval", "vector search", "vector database", "faiss",
    "pinecone", "weaviate", "qdrant", "milvus", "elasticsearch", "opensearch",
    "ranking system", "recommendation system", "recommender system",
    "search relevance", "semantic search", "hybrid search", "re-rank",
    "reranking", "ndcg", "mrr", "learning to rank", "ann index",
    "approximate nearest neighbor",
]

RESEARCH_ONLY_KEYWORDS = [
    "research scientist", "phd researcher", "academic lab", "research fellow",
    "postdoc", "research intern", "publication-only",
]

CV_SPEECH_ROBOTICS_KEYWORDS = [
    "computer vision", "speech recognition", "robotics", "autonomous driving",
    "lidar", "slam", "image classification", "object detection",
]

NLP_IR_KEYWORDS = [
    "nlp", "natural language processing", "information retrieval", "llm",
    "language model", "text classification", "embeddings", "transformer",
    "bert", "gpt", "search", "ranking",
]

FRAMEWORK_TUTORIAL_KEYWORDS = [
    "langchain tutorial", "how i used", "demo project", "toy project",
    "followed a tutorial", "built a demo",
]

RECENT_LANGCHAIN_ONLY_KEYWORDS = ["langchain", "openai api", "prompt engineering"]


def load_candidates(path):
    """Yields parsed candidate dicts. Handles both .jsonl and .jsonl.gz."""
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def _safe_lower(x):
    return (x or "").lower()


def candidate_text(c):
    """Build the text blob used for embedding. Weights recent roles higher by repetition."""
    profile = c.get("profile", {})
    parts = [
        profile.get("headline", ""),
        profile.get("summary", ""),
        profile.get("current_title", ""),
        profile.get("current_title", ""),  # repeat current title -> extra weight
    ]
    career = c.get("career_history", []) or []
    # sort most-recent first (is_current, then start_date desc)
    def sort_key(entry):
        return (not entry.get("is_current", False), entry.get("start_date", "") or "")
    career_sorted = sorted(career, key=sort_key)
    for i, entry in enumerate(career_sorted[:5]):
        weight = 2 if i == 0 else 1  # most recent role weighted x2
        text = f"{entry.get('title','')} at {entry.get('company','')}: {entry.get('description','')}"
        parts.extend([text] * weight)

    skills = c.get("skills", []) or []
    # weight skills by endorsements + duration, so lazily-listed skills contribute less
    skill_terms = []
    for s in sorted(skills, key=lambda s: (s.get("endorsements", 0), s.get("duration_months", 0)), reverse=True)[:15]:
        if s.get("duration_months", 0) >= 6:  # only include skills with real usage evidence
            skill_terms.append(s.get("name", ""))
    parts.append(" ".join(skill_terms))

    return " ".join(p for p in parts if p)


def years_experience_fit(years):
    """Soft curve centered on 5-9 yrs. Returns multiplier 0.55-1.0."""
    if years is None:
        return 0.7
    if 5 <= years <= 9:
        return 1.0
    if years < 5:
        gap = 5 - years
        return max(0.55, 1.0 - 0.12 * gap)
    gap = years - 9
    return max(0.6, 1.0 - 0.08 * gap)


def location_fit(location, country):
    loc = _safe_lower(location)
    country = _safe_lower(country)
    if country and country != "india":
        return 0.75  # outside India: case-by-case, no visa sponsorship
    for p in PREFERRED_LOCATIONS:
        if p in loc:
            return 1.0
    for a in ACCEPTABLE_LOCATIONS:
        if a in loc:
            return 0.9
    return 0.8


def is_pure_consulting_career(career):
    if not career:
        return False
    companies = [_safe_lower(e.get("company", "")) for e in career]
    return all(any(cf in comp for cf in CONSULTING_FIRMS) for comp in companies) and len(companies) > 0


def avg_tenure_months(career):
    durations = [e.get("duration_months", 0) for e in career if e.get("duration_months")]
    if not durations:
        return None
    return sum(durations) / len(durations)


def career_blob(c):
    career = c.get("career_history", []) or []
    parts = [f"{e.get('title','')} {e.get('description','')}" for e in career]
    parts.append(c.get("profile", {}).get("summary", ""))
    parts.append(c.get("profile", {}).get("headline", ""))
    return _safe_lower(" ".join(parts))


def has_retrieval_evidence(c):
    blob = career_blob(c)
    return sum(1 for k in RETRIEVAL_KEYWORDS if k in blob)


def hard_penalty_gate(c):
    """
    Returns a multiplier 0.0-1.0 encoding the JD's explicit "do NOT want" list.
    Multiple violations stack multiplicatively.
    """
    penalty = 1.0
    career = c.get("career_history", []) or []
    blob = career_blob(c)
    profile = c.get("profile", {})
    title = _safe_lower(profile.get("current_title", ""))

    # Pure research-only background, no production deployment
    research_hits = sum(1 for k in RESEARCH_ONLY_KEYWORDS if k in blob)
    has_production_evidence = any(w in blob for w in ["deployed", "production", "shipped", "launched", "scale"])
    if research_hits > 0 and not has_production_evidence:
        penalty *= 0.2

    # 100% consulting-firm career with no product company experience
    if is_pure_consulting_career(career):
        penalty *= 0.25

    # CV/speech/robotics-only, no NLP/IR exposure
    cv_hits = sum(1 for k in CV_SPEECH_ROBOTICS_KEYWORDS if k in blob)
    nlp_hits = sum(1 for k in NLP_IR_KEYWORDS if k in blob)
    if cv_hits >= 2 and nlp_hits == 0:
        penalty *= 0.3

    # Title-hopping: avg tenure well under 18 months across a multi-role career
    tenure = avg_tenure_months(career)
    if tenure is not None and len(career) >= 3 and tenure < 18:
        penalty *= 0.5

    # Framework-tutorial-only signal: keywords present but no retrieval evidence
    tut_hits = sum(1 for k in FRAMEWORK_TUTORIAL_KEYWORDS if k in blob)
    if tut_hits > 0 and has_retrieval_evidence(c) == 0:
        penalty *= 0.4

    # Recent-only LangChain/OpenAI-wrapper experience, no pre-LLM-era production ML
    skills = c.get("skills", []) or []
    skill_names = {_safe_lower(s.get("name", "")) for s in skills}
    langchain_only = skill_names & set(RECENT_LANGCHAIN_ONLY_KEYWORDS)
    pre_llm_evidence = any(w in blob for w in ["spark", "recommendation", "search", "ranking", "retrieval", "ml pipeline", "machine learning"])
    if langchain_only and not pre_llm_evidence and has_retrieval_evidence(c) == 0:
        penalty *= 0.35

    # Not senior-tech-lead-who-doesn't-code: penalize pure management/architecture titles w/ no recent hands-on
    if any(t in title for t in ["marketing manager", "hr manager", "sales manager", "operations manager", "customer support", "content writer"]):
        penalty *= 0.05  # clearly off-track roles regardless of skills listed

    return max(penalty, 0.02)


def is_honeypot(c):
    """Detect subtly impossible profiles."""
    career = c.get("career_history", []) or []
    skills = c.get("skills", []) or []
    profile = c.get("profile", {})

    # expert proficiency with 0 (or near-0) duration_months used
    for s in skills:
        if s.get("proficiency") == "expert" and s.get("duration_months", 0) == 0:
            return True

    # years_of_experience wildly inconsistent with sum of career durations
    yoe = profile.get("years_of_experience")
    total_months = sum(e.get("duration_months", 0) for e in career)
    if yoe is not None and total_months > 0:
        implied_years = total_months / 12.0
        if yoe > implied_years * 2.5 and yoe - implied_years > 3:
            return True

    # overlapping career_history date ranges (more than one "is_current" or overlapping spans)
    current_count = sum(1 for e in career if e.get("is_current"))
    if current_count > 1:
        return True

    parsed = []
    for e in career:
        try:
            sd = datetime.strptime(e["start_date"][:10], "%Y-%m-%d").date()
        except Exception:
            continue
        ed_raw = e.get("end_date")
        ed = date.today() if not ed_raw else datetime.strptime(ed_raw[:10], "%Y-%m-%d").date()
        parsed.append((sd, ed))
    parsed.sort()
    for i in range(len(parsed) - 1):
        if parsed[i][1] > parsed[i + 1][0]:
            # allow small overlap (<45 days, common in real data for transitions)
            if (parsed[i][1] - parsed[i + 1][0]).days > 45:
                return True

    return False


def behavioral_multiplier(c):
    """0.4-1.0 based on the 23 redrob_signals."""
    sig = c.get("redrob_signals", {}) or {}
    mult = 1.0

    # recency of activity
    last_active = sig.get("last_active_date")
    if last_active:
        try:
            la = datetime.strptime(last_active[:10], "%Y-%m-%d").date()
            days_inactive = (date.today() - la).days
            if days_inactive > 180:
                mult *= 0.5
            elif days_inactive > 90:
                mult *= 0.75
            elif days_inactive > 30:
                mult *= 0.9
        except Exception:
            pass

    resp_rate = sig.get("recruiter_response_rate")
    if resp_rate is not None:
        mult *= (0.6 + 0.4 * resp_rate)  # 0.6-1.0

    interview_rate = sig.get("interview_completion_rate")
    if interview_rate is not None:
        mult *= (0.8 + 0.2 * interview_rate)  # 0.8-1.0

    open_to_work = sig.get("open_to_work_flag")
    if open_to_work is False:
        mult *= 0.85

    completeness = sig.get("profile_completeness_score")
    if completeness is not None:
        mult *= (0.85 + 0.15 * (completeness / 100.0))  # 0.85-1.0

    return max(mult, 0.3)


def build_reasoning(c, sim_score, penalty, behav_mult, retrieval_hits):
    profile = c.get("profile", {})
    yoe = profile.get("years_of_experience")
    title = profile.get("current_title", "")
    company = profile.get("current_company", "")
    location = profile.get("location", "")
    sig = c.get("redrob_signals", {}) or {}
    resp_rate = sig.get("recruiter_response_rate")

    bits = [f"{title} with {yoe} yrs, currently at {company} ({location})."]
    if retrieval_hits > 0:
        bits.append(f"Career history shows {retrieval_hits} direct signal(s) of ranking/retrieval/search work.")
    else:
        bits.append("Limited direct ranking/retrieval evidence in career history.")
    if penalty < 0.5:
        bits.append("Some concerns flagged against the JD's explicit disqualifiers.")
    if resp_rate is not None:
        bits.append(f"Recruiter response rate {resp_rate:.0%}.")
    return " ".join(bits)



In [3]:
# Step 3: load the pre loaded candidate sample
# This is a small slice (50 candidates) of sample_candidates.json from the
# hackathon bundle. No upload step needed, it is embedded directly here.

import json

sample_candidates = json.loads('''[{"candidate_id": "CAND_0000001", "profile": {"anonymized_name": "Ira Vora", "headline": "Backend Engineer | SQL, Spark, Cloud", "summary": "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid \u2014 Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side \u2014 Python, SQL, Spark, Airflow, warehouse design \u2014 and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.", "location": "Toronto", "country": "Canada", "years_of_experience": 6.9, "current_title": "Backend Engineer", "current_company": "Mindtree", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Mindtree", "title": "Backend Engineer", "start_date": "2024-03-08", "end_date": null, "duration_months": 27, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure."}, {"company": "Dunder Mifflin", "title": "Analytics Engineer", "start_date": "2019-07-03", "end_date": "2024-01-08", "duration_months": 55, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Built and maintained data pipelines on Apache Airflow processing ~500GB of daily transactional data across 12 source systems. Worked extensively with Spark (PySpark) for batch processing and dbt for the transformation/modeling layer in our Snowflake warehouse. Owned the on-call rotation for data quality issues \u2014 wrote most of the data quality checks that detect schema drift and unusual volume changes. The pipeline supports the analytics team and a few internal ML models."}], "education": [{"institution": "Lovely Professional University", "degree": "B.E.", "field_of_study": "Computer Science", "start_year": 2017, "end_year": 2020, "grade": "8.24 CGPA", "tier": "tier_3"}], "skills": [{"name": "Tailwind", "proficiency": "intermediate", "endorsements": 3, "duration_months": 13}, {"name": "NLP", "proficiency": "advanced", "endorsements": 37, "duration_months": 26}, {"name": "Image Classification", "proficiency": "advanced", "endorsements": 7, "duration_months": 40}, {"name": "Fine-tuning LLMs", "proficiency": "advanced", "endorsements": 21, "duration_months": 36}, {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 13, "duration_months": 30}, {"name": "Speech Recognition", "proficiency": "advanced", "endorsements": 52, "duration_months": 33}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 8, "duration_months": 24}, {"name": "TTS", "proficiency": "advanced", "endorsements": 56, "duration_months": 60}, {"name": "LoRA", "proficiency": "intermediate", "endorsements": 0, "duration_months": 28}, {"name": "Apache Beam", "proficiency": "intermediate", "endorsements": 4, "duration_months": 9}, {"name": "AWS", "proficiency": "beginner", "endorsements": 5, "duration_months": 8}, {"name": "Flask", "proficiency": "beginner", "endorsements": 15, "duration_months": 15}, {"name": "BentoML", "proficiency": "intermediate", "endorsements": 3, "duration_months": 36}, {"name": "Milvus", "proficiency": "advanced", "endorsements": 40, "duration_months": 35}, {"name": "GANs", "proficiency": "advanced", "endorsements": 12, "duration_months": 19}, {"name": "Statistical Modeling", "proficiency": "intermediate", "endorsements": 9, "duration_months": 8}, {"name": "GCP", "proficiency": "beginner", "endorsements": 7, "duration_months": 2}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 86.9, "signup_date": "2025-10-16", "last_active_date": "2026-05-20", "open_to_work_flag": true, "profile_views_received_30d": 23, "applications_submitted_30d": 2, "recruiter_response_rate": 0.34, "avg_response_time_hours": 177.8, "skill_assessment_scores": {"NLP": 38.8, "Image Classification": 64.8, "Fine-tuning LLMs": 41.6, "Speech Recognition": 53.7}, "connection_count": 356, "endorsements_received": 35, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 18.7, "max": 36.1}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": 9.2, "search_appearance_30d": 249, "saved_by_recruiters_30d": 4, "interview_completion_rate": 0.71, "offer_acceptance_rate": 0.58, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000002", "profile": {"anonymized_name": "Saanvi Sethi", "headline": "Operations Manager | 12.5+ yrs experience", "summary": "Professional with 12.5+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Chennai, Tamil Nadu", "country": "India", "years_of_experience": 12.5, "current_title": "Operations Manager", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Operations Manager", "start_date": "2022-11-14", "end_date": null, "duration_months": 43, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Wipro", "title": "Operations Manager", "start_date": "2021-09-13", "end_date": "2022-11-07", "duration_months": 14, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Acme Corp", "title": "Marketing Manager", "start_date": "2017-03-08", "end_date": "2021-08-14", "duration_months": 54, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}, {"company": "Dunder Mifflin", "title": "Marketing Manager", "start_date": "2014-01-23", "end_date": "2017-03-08", "duration_months": 38, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "Local Engineering College", "degree": "B.Sc", "field_of_study": "Mathematics", "start_year": 2007, "end_year": 2011, "grade": "77%", "tier": "tier_4"}], "skills": [{"name": "Project Management", "proficiency": "intermediate", "endorsements": 14, "duration_months": 23}, {"name": "React", "proficiency": "intermediate", "endorsements": 6, "duration_months": 35}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 9, "duration_months": 35}, {"name": "TypeScript", "proficiency": "beginner", "endorsements": 2, "duration_months": 3}, {"name": "Marketing", "proficiency": "beginner", "endorsements": 9, "duration_months": 11}, {"name": "Kafka", "proficiency": "intermediate", "endorsements": 3, "duration_months": 16}, {"name": "JavaScript", "proficiency": "beginner", "endorsements": 9, "duration_months": 3}, {"name": "Feature Engineering", "proficiency": "intermediate", "endorsements": 11, "duration_months": 27}, {"name": "GCP", "proficiency": "intermediate", "endorsements": 7, "duration_months": 30}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 78.7, "signup_date": "2025-07-28", "last_active_date": "2025-11-12", "open_to_work_flag": true, "profile_views_received_30d": 7, "applications_submitted_30d": 1, "recruiter_response_rate": 0.29, "avg_response_time_hours": 171.6, "skill_assessment_scores": {}, "connection_count": 179, "endorsements_received": 3, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 8.8, "max": 9.0}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 107, "saved_by_recruiters_30d": 10, "interview_completion_rate": 0.62, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000003", "profile": {"anonymized_name": "Yash Agarwal", "headline": "Customer Support | 1.1+ yrs experience", "summary": "Professional with 1.1+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Austin", "country": "USA", "years_of_experience": 1.1, "current_title": "Customer Support", "current_company": "TCS", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "TCS", "title": "Customer Support", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}], "education": [{"institution": "Local Engineering College", "degree": "M.E.", "field_of_study": "Chemical Engineering", "start_year": 2005, "end_year": 2010, "grade": "6.82 CGPA", "tier": "tier_4"}, {"institution": "Chandigarh University", "degree": "M.Sc", "field_of_study": "Information Technology", "start_year": 2017, "end_year": 2021, "grade": "87%", "tier": "tier_3"}], "skills": [{"name": "Angular", "proficiency": "intermediate", "endorsements": 13, "duration_months": 10}, {"name": "SEO", "proficiency": "beginner", "endorsements": 11, "duration_months": 11}, {"name": "Excel", "proficiency": "intermediate", "endorsements": 2, "duration_months": 15}, {"name": "Accounting", "proficiency": "beginner", "endorsements": 7, "duration_months": 18}, {"name": "Kubernetes", "proficiency": "intermediate", "endorsements": 0, "duration_months": 34}, {"name": "Databricks", "proficiency": "beginner", "endorsements": 14, "duration_months": 18}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 31.9, "signup_date": "2024-08-02", "last_active_date": "2026-03-21", "open_to_work_flag": false, "profile_views_received_30d": 1, "applications_submitted_30d": 9, "recruiter_response_rate": 0.46, "avg_response_time_hours": 119.4, "skill_assessment_scores": {}, "connection_count": 19, "endorsements_received": 46, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 11.2, "max": 18.1}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 28, "saved_by_recruiters_30d": 4, "interview_completion_rate": 0.86, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000004", "profile": {"anonymized_name": "Anil Bose", "headline": "Marketing Manager | Driving business outcomes", "summary": "Professional with 3.8+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Sydney", "country": "Australia", "years_of_experience": 3.8, "current_title": "Marketing Manager", "current_company": "Dunder Mifflin", "current_company_size": "201-500", "current_industry": "Paper Products"}, "career_history": [{"company": "Dunder Mifflin", "title": "Marketing Manager", "start_date": "2025-04-02", "end_date": null, "duration_months": 14, "is_current": true, "industry": "Paper Products", "company_size": "201-500", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Infosys", "title": "Operations Manager", "start_date": "2023-07-28", "end_date": "2025-03-19", "duration_months": 20, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}, {"company": "Globex Inc", "title": "Business Analyst", "start_date": "2022-08-02", "end_date": "2023-05-29", "duration_months": 10, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}], "education": [{"institution": "Local Engineering College", "degree": "B.Tech", "field_of_study": "Machine Learning", "start_year": 2015, "end_year": 2019, "grade": "7.72 CGPA", "tier": "tier_4"}, {"institution": "Lovely Professional University", "degree": "Ph.D", "field_of_study": "Electronics", "start_year": 2013, "end_year": 2016, "grade": "7.61 CGPA", "tier": "tier_3"}], "skills": [{"name": "Node.js", "proficiency": "intermediate", "endorsements": 1, "duration_months": 20}, {"name": "Content Writing", "proficiency": "beginner", "endorsements": 7, "duration_months": 3}, {"name": "Redux", "proficiency": "beginner", "endorsements": 14, "duration_months": 8}, {"name": "Airflow", "proficiency": "intermediate", "endorsements": 11, "duration_months": 27}, {"name": "GraphQL", "proficiency": "beginner", "endorsements": 13, "duration_months": 2}, {"name": "Object Detection", "proficiency": "intermediate", "endorsements": 3, "duration_months": 17}, {"name": "Webpack", "proficiency": "beginner", "endorsements": 10, "duration_months": 7}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 1, "duration_months": 12}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 6, "duration_months": 9}, {"name": "JavaScript", "proficiency": "intermediate", "endorsements": 14, "duration_months": 29}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2025}, {"name": "Scrum Master Certified", "issuer": "Scrum Alliance", "year": 2025}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 28.5, "signup_date": "2025-07-21", "last_active_date": "2026-03-25", "open_to_work_flag": false, "profile_views_received_30d": 3, "applications_submitted_30d": 9, "recruiter_response_rate": 0.26, "avg_response_time_hours": 104.1, "skill_assessment_scores": {}, "connection_count": 485, "endorsements_received": 22, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 4.6, "max": 6.7}, "preferred_work_mode": "onsite", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 5, "saved_by_recruiters_30d": 8, "interview_completion_rate": 0.35, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000005", "profile": {"anonymized_name": "Aisha Sethi", "headline": "Accountant | Helping teams scale", "summary": "Professional with 11.0+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Gurgaon, Haryana", "country": "India", "years_of_experience": 11.0, "current_title": "Accountant", "current_company": "Stark Industries", "current_company_size": "1001-5000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Stark Industries", "title": "Accountant", "start_date": "2022-02-17", "end_date": null, "duration_months": 52, "is_current": true, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Wipro", "title": "HR Manager", "start_date": "2018-05-25", "end_date": "2022-02-03", "duration_months": 45, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Initech", "title": "HR Manager", "start_date": "2016-06-04", "end_date": "2018-05-25", "duration_months": 24, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "TCS", "title": "Accountant", "start_date": "2015-09-08", "end_date": "2016-06-04", "duration_months": 9, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}], "education": [{"institution": "Chandigarh University", "degree": "M.Sc", "field_of_study": "Information Technology", "start_year": 2007, "end_year": 2012, "grade": "87%", "tier": "tier_3"}], "skills": [{"name": "SQL", "proficiency": "beginner", "endorsements": 12, "duration_months": 12}, {"name": "PowerPoint", "proficiency": "beginner", "endorsements": 6, "duration_months": 14}, {"name": "Photoshop", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "Tailwind", "proficiency": "intermediate", "endorsements": 15, "duration_months": 35}, {"name": "Apache Flink", "proficiency": "intermediate", "endorsements": 1, "duration_months": 30}, {"name": "Image Classification", "proficiency": "advanced", "endorsements": 50, "duration_months": 38}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 84.6, "signup_date": "2023-10-07", "last_active_date": "2025-10-01", "open_to_work_flag": true, "profile_views_received_30d": 12, "applications_submitted_30d": 2, "recruiter_response_rate": 0.37, "avg_response_time_hours": 116.7, "skill_assessment_scores": {}, "connection_count": 300, "endorsements_received": 14, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 12.4, "max": 19.7}, "preferred_work_mode": "hybrid", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 67, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.74, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000006", "profile": {"anonymized_name": "Rajesh Desai", "headline": "Business Analyst | 6.0+ yrs experience", "summary": "Professional with 6.0+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Austin", "country": "USA", "years_of_experience": 6.0, "current_title": "Business Analyst", "current_company": "Wayne Enterprises", "current_company_size": "10001+", "current_industry": "Conglomerate"}, "career_history": [{"company": "Wayne Enterprises", "title": "Business Analyst", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "Conglomerate", "company_size": "10001+", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Pied Piper", "title": "Mechanical Engineer", "start_date": "2020-07-27", "end_date": "2023-09-10", "duration_months": 38, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}], "education": [{"institution": "Lovely Professional University", "degree": "B.Sc", "field_of_study": "Artificial Intelligence", "start_year": 2005, "end_year": 2008, "grade": "9.26 CGPA", "tier": "tier_3"}], "skills": [{"name": "Content Writing", "proficiency": "intermediate", "endorsements": 0, "duration_months": 33}, {"name": "SEO", "proficiency": "intermediate", "endorsements": 13, "duration_months": 31}, {"name": "Redux", "proficiency": "beginner", "endorsements": 15, "duration_months": 12}, {"name": "SQL", "proficiency": "beginner", "endorsements": 9, "duration_months": 11}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 5, "duration_months": 27}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 8, "duration_months": 3}, {"name": "Django", "proficiency": "intermediate", "endorsements": 3, "duration_months": 11}, {"name": "Terraform", "proficiency": "beginner", "endorsements": 4, "duration_months": 13}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 29.7, "signup_date": "2026-04-26", "last_active_date": "2026-02-28", "open_to_work_flag": false, "profile_views_received_30d": 53, "applications_submitted_30d": 8, "recruiter_response_rate": 0.12, "avg_response_time_hours": 172.1, "skill_assessment_scores": {}, "connection_count": 389, "endorsements_received": 29, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 7.7, "max": 11.7}, "preferred_work_mode": "remote", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 131, "saved_by_recruiters_30d": 9, "interview_completion_rate": 0.57, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000007", "profile": {"anonymized_name": "Vihaan Bose", "headline": "Civil Engineer | 5.5+ yrs experience", "summary": "Professional with 5.5+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Gurgaon, Haryana", "country": "India", "years_of_experience": 5.5, "current_title": "Civil Engineer", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Civil Engineer", "start_date": "2023-04-13", "end_date": null, "duration_months": 38, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}, {"company": "Initech", "title": "Mechanical Engineer", "start_date": "2021-01-16", "end_date": "2023-04-06", "duration_months": 27, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}], "education": [{"institution": "SRM University", "degree": "M.E.", "field_of_study": "Data Science", "start_year": 2009, "end_year": 2013, "grade": "8.28 CGPA", "tier": "tier_2"}], "skills": [{"name": "Content Writing", "proficiency": "beginner", "endorsements": 12, "duration_months": 14}, {"name": "MongoDB", "proficiency": "intermediate", "endorsements": 13, "duration_months": 9}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 0, "duration_months": 27}, {"name": "Spark", "proficiency": "beginner", "endorsements": 14, "duration_months": 14}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "Apache Beam", "proficiency": "beginner", "endorsements": 4, "duration_months": 3}, {"name": "Illustrator", "proficiency": "beginner", "endorsements": 14, "duration_months": 2}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 74.6, "signup_date": "2025-09-29", "last_active_date": "2026-05-25", "open_to_work_flag": false, "profile_views_received_30d": 2, "applications_submitted_30d": 1, "recruiter_response_rate": 0.62, "avg_response_time_hours": 61.3, "skill_assessment_scores": {}, "connection_count": 122, "endorsements_received": 50, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 6.7, "max": 14.6}, "preferred_work_mode": "onsite", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 104, "saved_by_recruiters_30d": 8, "interview_completion_rate": 0.47, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000008", "profile": {"anonymized_name": "Shaurya Chatterjee", "headline": "Operations Manager | 3.6+ yrs experience", "summary": "Professional with 3.6+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Noida, Uttar Pradesh", "country": "India", "years_of_experience": 3.6, "current_title": "Operations Manager", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Operations Manager", "start_date": "2022-11-14", "end_date": null, "duration_months": 43, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}], "education": [{"institution": "Anna University", "degree": "B.Tech", "field_of_study": "Data Science", "start_year": 2008, "end_year": 2012, "grade": "8.60 CGPA", "tier": "tier_2"}, {"institution": "SRM University", "degree": "M.Sc", "field_of_study": "Computer Engineering", "start_year": 2009, "end_year": 2013, "grade": "67%", "tier": "tier_2"}], "skills": [{"name": "Java", "proficiency": "intermediate", "endorsements": 2, "duration_months": 32}, {"name": "BigQuery", "proficiency": "beginner", "endorsements": 5, "duration_months": 9}, {"name": "Spark", "proficiency": "beginner", "endorsements": 4, "duration_months": 6}, {"name": "Accounting", "proficiency": "beginner", "endorsements": 8, "duration_months": 3}, {"name": "Kubernetes", "proficiency": "intermediate", "endorsements": 2, "duration_months": 9}, {"name": "TypeScript", "proficiency": "intermediate", "endorsements": 14, "duration_months": 11}, {"name": "Rust", "proficiency": "intermediate", "endorsements": 12, "duration_months": 16}, {"name": "HTML", "proficiency": "beginner", "endorsements": 8, "duration_months": 11}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2018}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 63.0, "signup_date": "2022-06-26", "last_active_date": "2025-12-13", "open_to_work_flag": false, "profile_views_received_30d": 28, "applications_submitted_30d": 5, "recruiter_response_rate": 0.42, "avg_response_time_hours": 98.4, "skill_assessment_scores": {}, "connection_count": 285, "endorsements_received": 7, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 6.6, "max": 17.2}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 91, "saved_by_recruiters_30d": 0, "interview_completion_rate": 0.74, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000009", "profile": {"anonymized_name": "Amit Shah", "headline": "Mechanical Engineer | Driving business outcomes", "summary": "Professional with 11.0+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "New York", "country": "USA", "years_of_experience": 11.0, "current_title": "Mechanical Engineer", "current_company": "Dunder Mifflin", "current_company_size": "201-500", "current_industry": "Paper Products"}, "career_history": [{"company": "Dunder Mifflin", "title": "Mechanical Engineer", "start_date": "2022-10-15", "end_date": null, "duration_months": 44, "is_current": true, "industry": "Paper Products", "company_size": "201-500", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Wipro", "title": "Content Writer", "start_date": "2021-02-22", "end_date": "2022-10-15", "duration_months": 20, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Stark Industries", "title": "Customer Support", "start_date": "2017-02-13", "end_date": "2021-02-22", "duration_months": 49, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Acme Corp", "title": "Project Manager", "start_date": "2015-08-23", "end_date": "2017-02-13", "duration_months": 18, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}], "education": [{"institution": "KIIT University", "degree": "B.Tech", "field_of_study": "Electronics", "start_year": 2009, "end_year": 2014, "grade": "7.89 CGPA", "tier": "tier_3"}], "skills": [{"name": "Snowflake", "proficiency": "intermediate", "endorsements": 5, "duration_months": 8}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 6, "duration_months": 12}, {"name": "JavaScript", "proficiency": "intermediate", "endorsements": 0, "duration_months": 23}, {"name": "OpenCV", "proficiency": "intermediate", "endorsements": 12, "duration_months": 36}, {"name": "Go", "proficiency": "intermediate", "endorsements": 12, "duration_months": 20}, {"name": "PowerPoint", "proficiency": "intermediate", "endorsements": 1, "duration_months": 10}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 39.6, "signup_date": "2025-10-19", "last_active_date": "2026-01-27", "open_to_work_flag": false, "profile_views_received_30d": 50, "applications_submitted_30d": 8, "recruiter_response_rate": 0.53, "avg_response_time_hours": 202.0, "skill_assessment_scores": {}, "connection_count": 516, "endorsements_received": 34, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 16.0, "max": 7.3}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 74, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.54, "offer_acceptance_rate": 0.48, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000010", "profile": {"anonymized_name": "Aarav Kapoor", "headline": "Data Engineer | Data pipelines & analytics", "summary": "Software / data professional with 4.6 years of experience building data pipelines, backend systems, and analytics infrastructure. Most of my work has been data pipelines and analytics infrastructure; I've shipped a couple of small ML features but it's not the core of my day. My toolkit is solid on the data engineering side \u2014 Python, SQL, Spark, Airflow, warehouse design \u2014 and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.", "location": "London", "country": "UK", "years_of_experience": 4.6, "current_title": "Data Engineer", "current_company": "Ola", "current_company_size": "5001-10000", "current_industry": "Transportation"}, "career_history": [{"company": "Ola", "title": "Data Engineer", "start_date": "2021-11-19", "end_date": null, "duration_months": 55, "is_current": true, "industry": "Transportation", "company_size": "5001-10000", "description": "Mixed data science and analytics-engineering role at a marketing-analytics startup. Spent maybe 30% of my time on lightweight ML (clustering, classification, churn prediction in sklearn/XGBoost) and 70% on data infrastructure and dashboards. Comfortable with the modeling work but I wouldn't call myself an ML specialist. Built our experimentation framework that supports the product team's A/B tests."}], "education": [{"institution": "Generic State University", "degree": "B.E.", "field_of_study": "Mathematics", "start_year": 2007, "end_year": 2011, "grade": "85%", "tier": "tier_4"}, {"institution": "Local Engineering College", "degree": "M.S.", "field_of_study": "Computer Engineering", "start_year": 2013, "end_year": 2018, "grade": "7.73 CGPA", "tier": "tier_4"}], "skills": [{"name": "GCP", "proficiency": "beginner", "endorsements": 7, "duration_months": 8}, {"name": "Spring Boot", "proficiency": "beginner", "endorsements": 3, "duration_months": 2}, {"name": "Kubeflow", "proficiency": "intermediate", "endorsements": 11, "duration_months": 19}, {"name": "Java", "proficiency": "intermediate", "endorsements": 12, "duration_months": 19}, {"name": "GANs", "proficiency": "advanced", "endorsements": 58, "duration_months": 57}, {"name": "Figma", "proficiency": "beginner", "endorsements": 4, "duration_months": 3}, {"name": "Elasticsearch", "proficiency": "intermediate", "endorsements": 15, "duration_months": 17}, {"name": "OpenCV", "proficiency": "advanced", "endorsements": 0, "duration_months": 24}, {"name": "CNN", "proficiency": "intermediate", "endorsements": 15, "duration_months": 8}, {"name": "Azure", "proficiency": "beginner", "endorsements": 7, "duration_months": 11}, {"name": "Prompt Engineering", "proficiency": "advanced", "endorsements": 42, "duration_months": 35}, {"name": "Object Detection", "proficiency": "advanced", "endorsements": 55, "duration_months": 58}, {"name": "MLOps", "proficiency": "intermediate", "endorsements": 3, "duration_months": 10}, {"name": "Python", "proficiency": "intermediate", "endorsements": 7, "duration_months": 14}, {"name": "BM25", "proficiency": "advanced", "endorsements": 55, "duration_months": 55}, {"name": "Weights & Biases", "proficiency": "advanced", "endorsements": 4, "duration_months": 21}, {"name": "Sales", "proficiency": "beginner", "endorsements": 5, "duration_months": 18}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 81.6, "signup_date": "2026-01-09", "last_active_date": "2026-04-29", "open_to_work_flag": false, "profile_views_received_30d": 60, "applications_submitted_30d": 13, "recruiter_response_rate": 0.4, "avg_response_time_hours": 19.0, "skill_assessment_scores": {"GANs": 53.3, "OpenCV": 65.5, "Prompt Engineering": 73.8, "Object Detection": 81.3}, "connection_count": 712, "endorsements_received": 38, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 13.0, "max": 32.0}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": 33.7, "search_appearance_30d": 256, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.53, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000011", "profile": {"anonymized_name": "Deepak Desai", "headline": "QA Engineer | Cloud & DevOps", "summary": "Software engineer with 2.0 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering \u2014 APIs, databases, queues, deployments. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Hyderabad, Telangana", "country": "India", "years_of_experience": 2.0, "current_title": "QA Engineer", "current_company": "Pied Piper", "current_company_size": "11-50", "current_industry": "Software"}, "career_history": [{"company": "Pied Piper", "title": "QA Engineer", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "Software", "company_size": "11-50", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Hooli", "title": "QA Engineer", "start_date": "2024-06-06", "end_date": "2025-04-02", "duration_months": 10, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}], "education": [{"institution": "Chandigarh University", "degree": "B.Tech", "field_of_study": "Data Science", "start_year": 2014, "end_year": 2019, "grade": "6.96 CGPA", "tier": "tier_3"}, {"institution": "Anna University", "degree": "B.Sc", "field_of_study": "Information Technology", "start_year": 2015, "end_year": 2020, "grade": "9.16 CGPA", "tier": "tier_2"}], "skills": [{"name": "Recommendation Systems", "proficiency": "advanced", "endorsements": 5, "duration_months": 40}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 13, "duration_months": 7}, {"name": "FastAPI", "proficiency": "beginner", "endorsements": 4, "duration_months": 12}, {"name": "Hugging Face Transformers", "proficiency": "intermediate", "endorsements": 1, "duration_months": 30}, {"name": "AWS", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "Snowflake", "proficiency": "beginner", "endorsements": 4, "duration_months": 11}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 12, "duration_months": 31}, {"name": "PostgreSQL", "proficiency": "intermediate", "endorsements": 7, "duration_months": 24}, {"name": "Kubeflow", "proficiency": "advanced", "endorsements": 6, "duration_months": 59}, {"name": "Azure", "proficiency": "beginner", "endorsements": 12, "duration_months": 8}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2019}, {"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2021}], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 59.2, "signup_date": "2023-07-22", "last_active_date": "2026-01-19", "open_to_work_flag": false, "profile_views_received_30d": 112, "applications_submitted_30d": 0, "recruiter_response_rate": 0.56, "avg_response_time_hours": 184.4, "skill_assessment_scores": {"Recommendation Systems": 29.8}, "connection_count": 496, "endorsements_received": 9, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 15.5, "max": 13.9}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": 32.3, "search_appearance_30d": 200, "saved_by_recruiters_30d": 13, "interview_completion_rate": 0.45, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000012", "profile": {"anonymized_name": "Anjali Krishnan", "headline": "Operations Manager | Driving business outcomes", "summary": "Professional with 1.1+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Chandigarh, Chandigarh", "country": "India", "years_of_experience": 1.1, "current_title": "Operations Manager", "current_company": "Stark Industries", "current_company_size": "1001-5000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Stark Industries", "title": "Operations Manager", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}], "education": [{"institution": "Symbiosis International", "degree": "B.Sc", "field_of_study": "Physics", "start_year": 2018, "end_year": 2022, "grade": "68%", "tier": "tier_3"}, {"institution": "Christ University", "degree": "B.Sc", "field_of_study": "Mechanical Engineering", "start_year": 2011, "end_year": 2015, "grade": "7.28 CGPA", "tier": "tier_3"}], "skills": [{"name": "Azure", "proficiency": "beginner", "endorsements": 7, "duration_months": 10}, {"name": "Airflow", "proficiency": "intermediate", "endorsements": 2, "duration_months": 15}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 15, "duration_months": 30}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 9, "duration_months": 2}, {"name": "Vue.js", "proficiency": "intermediate", "endorsements": 2, "duration_months": 15}, {"name": "dbt", "proficiency": "intermediate", "endorsements": 11, "duration_months": 22}, {"name": "Agile", "proficiency": "intermediate", "endorsements": 4, "duration_months": 14}, {"name": "PowerPoint", "proficiency": "intermediate", "endorsements": 1, "duration_months": 27}, {"name": "Content Writing", "proficiency": "intermediate", "endorsements": 3, "duration_months": 36}, {"name": "Project Management", "proficiency": "beginner", "endorsements": 5, "duration_months": 3}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 53.4, "signup_date": "2024-01-28", "last_active_date": "2025-10-28", "open_to_work_flag": false, "profile_views_received_30d": 60, "applications_submitted_30d": 1, "recruiter_response_rate": 0.16, "avg_response_time_hours": 38.4, "skill_assessment_scores": {}, "connection_count": 165, "endorsements_received": 31, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 14.8, "max": 7.9}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 61, "saved_by_recruiters_30d": 3, "interview_completion_rate": 0.42, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000013", "profile": {"anonymized_name": "Pari Nair", "headline": "Civil Engineer | Driving business outcomes", "summary": "Professional with 1.1+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Dubai", "country": "UAE", "years_of_experience": 1.1, "current_title": "Civil Engineer", "current_company": "Globex Inc", "current_company_size": "501-1000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Globex Inc", "title": "Civil Engineer", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "Manufacturing", "company_size": "501-1000", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}], "education": [{"institution": "Delhi College of Engineering", "degree": "B.E.", "field_of_study": "Information Technology", "start_year": 2019, "end_year": 2022, "grade": "8.84 CGPA", "tier": "tier_2"}, {"institution": "Amity University", "degree": "Ph.D", "field_of_study": "Information Technology", "start_year": 2008, "end_year": 2013, "grade": "8.29 CGPA", "tier": "tier_3"}], "skills": [{"name": "React", "proficiency": "intermediate", "endorsements": 5, "duration_months": 23}, {"name": "Redux", "proficiency": "intermediate", "endorsements": 11, "duration_months": 9}, {"name": "Vue.js", "proficiency": "beginner", "endorsements": 12, "duration_months": 12}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 1, "duration_months": 2}, {"name": "Spring Boot", "proficiency": "beginner", "endorsements": 12, "duration_months": 12}, {"name": "Spark", "proficiency": "intermediate", "endorsements": 7, "duration_months": 30}, {"name": "Data Pipelines", "proficiency": "intermediate", "endorsements": 6, "duration_months": 36}, {"name": "GCP", "proficiency": "intermediate", "endorsements": 0, "duration_months": 18}, {"name": "Flask", "proficiency": "intermediate", "endorsements": 12, "duration_months": 8}, {"name": "Snowflake", "proficiency": "intermediate", "endorsements": 2, "duration_months": 8}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 44.2, "signup_date": "2024-06-14", "last_active_date": "2026-03-26", "open_to_work_flag": true, "profile_views_received_30d": 16, "applications_submitted_30d": 3, "recruiter_response_rate": 0.28, "avg_response_time_hours": 80.3, "skill_assessment_scores": {}, "connection_count": 281, "endorsements_received": 9, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 11.6, "max": 8.1}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": 35.6, "search_appearance_30d": 40, "saved_by_recruiters_30d": 12, "interview_completion_rate": 0.33, "offer_acceptance_rate": 0.26, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000014", "profile": {"anonymized_name": "Atharv Joshi", "headline": "Frontend Engineer | Full-stack development", "summary": "Software engineer with 8.4 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Hyderabad, Telangana", "country": "India", "years_of_experience": 8.4, "current_title": "Frontend Engineer", "current_company": "Zomato", "current_company_size": "5001-10000", "current_industry": "Food Delivery"}, "career_history": [{"company": "Zomato", "title": "Frontend Engineer", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."}, {"company": "Dunder Mifflin", "title": "Software Engineer", "start_date": "2019-10-01", "end_date": "2023-09-10", "duration_months": 48, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Acme Corp", "title": "Java Developer", "start_date": "2018-03-03", "end_date": "2019-09-24", "duration_months": 19, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}], "education": [{"institution": "Lovely Professional University", "degree": "B.E.", "field_of_study": "Statistics", "start_year": 2012, "end_year": 2015, "grade": "7.45 CGPA", "tier": "tier_3"}], "skills": [{"name": "FAISS", "proficiency": "advanced", "endorsements": 40, "duration_months": 44}, {"name": "BigQuery", "proficiency": "intermediate", "endorsements": 6, "duration_months": 24}, {"name": "React", "proficiency": "beginner", "endorsements": 11, "duration_months": 10}, {"name": "OpenSearch", "proficiency": "advanced", "endorsements": 47, "duration_months": 59}, {"name": "OpenCV", "proficiency": "advanced", "endorsements": 30, "duration_months": 60}, {"name": "YOLO", "proficiency": "advanced", "endorsements": 1, "duration_months": 44}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 5, "duration_months": 30}, {"name": "SEO", "proficiency": "beginner", "endorsements": 0, "duration_months": 12}, {"name": "REST APIs", "proficiency": "beginner", "endorsements": 3, "duration_months": 4}, {"name": "GANs", "proficiency": "advanced", "endorsements": 9, "duration_months": 33}, {"name": "dbt", "proficiency": "beginner", "endorsements": 0, "duration_months": 13}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 1, "duration_months": 32}, {"name": "Tailwind", "proficiency": "intermediate", "endorsements": 2, "duration_months": 32}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 61.9, "signup_date": "2025-04-29", "last_active_date": "2026-04-12", "open_to_work_flag": false, "profile_views_received_30d": 21, "applications_submitted_30d": 1, "recruiter_response_rate": 0.8, "avg_response_time_hours": 7.7, "skill_assessment_scores": {"FAISS": 77.6}, "connection_count": 708, "endorsements_received": 63, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 9.0, "max": 30.0}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 12, "saved_by_recruiters_30d": 0, "interview_completion_rate": 0.55, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000015", "profile": {"anonymized_name": "Rahul Agarwal", "headline": "Software Engineer | Cloud & DevOps", "summary": "Software engineer with 5.4 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Trivandrum, Kerala", "country": "India", "years_of_experience": 5.4, "current_title": "Software Engineer", "current_company": "Razorpay", "current_company_size": "1001-5000", "current_industry": "Fintech"}, "career_history": [{"company": "Razorpay", "title": "Software Engineer", "start_date": "2023-11-09", "end_date": null, "duration_months": 31, "is_current": true, "industry": "Fintech", "company_size": "1001-5000", "description": "Cloud infrastructure and DevOps work at an enterprise SaaS company. Owned the AWS account architecture (VPC, IAM, networking), the Terraform modules for our service deployments, and the Kubernetes cluster operations. Designed the CI/CD pipelines (GitLab CI + ArgoCD) and the monitoring stack (Prometheus, Grafana, Loki). Strong on the infra and ops side; haven't done much application development."}, {"company": "Hooli", "title": "Mobile Developer", "start_date": "2021-11-12", "end_date": "2023-11-02", "duration_months": 24, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Globex Inc", "title": "DevOps Engineer", "start_date": "2021-02-15", "end_date": "2021-11-12", "duration_months": 9, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}], "education": [{"institution": "Local Engineering College", "degree": "Ph.D", "field_of_study": "Mathematics", "start_year": 2013, "end_year": 2017, "grade": "8.15 CGPA", "tier": "tier_4"}], "skills": [{"name": "PyTorch", "proficiency": "intermediate", "endorsements": 10, "duration_months": 15}, {"name": "Content Writing", "proficiency": "intermediate", "endorsements": 8, "duration_months": 31}, {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 5, "duration_months": 24}, {"name": "Qdrant", "proficiency": "intermediate", "endorsements": 13, "duration_months": 9}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 13, "duration_months": 30}, {"name": "Figma", "proficiency": "intermediate", "endorsements": 9, "duration_months": 29}, {"name": "BigQuery", "proficiency": "beginner", "endorsements": 0, "duration_months": 7}, {"name": "Computer Vision", "proficiency": "intermediate", "endorsements": 6, "duration_months": 20}, {"name": "Next.js", "proficiency": "intermediate", "endorsements": 12, "duration_months": 35}, {"name": "SEO", "proficiency": "intermediate", "endorsements": 10, "duration_months": 29}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 79.4, "signup_date": "2023-02-16", "last_active_date": "2026-02-12", "open_to_work_flag": true, "profile_views_received_30d": 93, "applications_submitted_30d": 3, "recruiter_response_rate": 0.32, "avg_response_time_hours": 117.7, "skill_assessment_scores": {}, "connection_count": 361, "endorsements_received": 61, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 21.8, "max": 18.9}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 164, "saved_by_recruiters_30d": 8, "interview_completion_rate": 0.72, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000016", "profile": {"anonymized_name": "Aanya Malhotra", "headline": "Accountant | Helping teams scale", "summary": "Professional with 5.3+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Gurgaon, Haryana", "country": "India", "years_of_experience": 5.3, "current_title": "Accountant", "current_company": "Infosys", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Infosys", "title": "Accountant", "start_date": "2024-12-03", "end_date": null, "duration_months": 18, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}, {"company": "TCS", "title": "Mechanical Engineer", "start_date": "2021-09-06", "end_date": "2024-11-19", "duration_months": 39, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Globex Inc", "title": "Operations Manager", "start_date": "2021-02-08", "end_date": "2021-08-07", "duration_months": 6, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}], "education": [{"institution": "Christ University", "degree": "B.E.", "field_of_study": "Electronics", "start_year": 2001, "end_year": 2006, "grade": "8.32 CGPA", "tier": "tier_3"}], "skills": [{"name": "Node.js", "proficiency": "beginner", "endorsements": 4, "duration_months": 11}, {"name": "Figma", "proficiency": "beginner", "endorsements": 12, "duration_months": 6}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 2, "duration_months": 7}, {"name": "Go", "proficiency": "beginner", "endorsements": 0, "duration_months": 10}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 10, "duration_months": 22}, {"name": "Kubeflow", "proficiency": "advanced", "endorsements": 2, "duration_months": 54}, {"name": "Accounting", "proficiency": "intermediate", "endorsements": 4, "duration_months": 31}, {"name": "SQL", "proficiency": "intermediate", "endorsements": 14, "duration_months": 16}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 69.4, "signup_date": "2022-12-12", "last_active_date": "2025-12-21", "open_to_work_flag": true, "profile_views_received_30d": 76, "applications_submitted_30d": 5, "recruiter_response_rate": 0.64, "avg_response_time_hours": 205.6, "skill_assessment_scores": {}, "connection_count": 148, "endorsements_received": 2, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 6.1, "max": 8.1}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": 42.9, "search_appearance_30d": 126, "saved_by_recruiters_30d": 5, "interview_completion_rate": 0.66, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0000017", "profile": {"anonymized_name": "Anil Pandey", "headline": "Accountant | 12.3+ yrs experience", "summary": "Professional with 12.3+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Bangalore, Karnataka", "country": "India", "years_of_experience": 12.3, "current_title": "Accountant", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Accountant", "start_date": "2024-03-08", "end_date": null, "duration_months": 27, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Infosys", "title": "Customer Support", "start_date": "2021-02-08", "end_date": "2024-02-23", "duration_months": 37, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Initech", "title": "Mechanical Engineer", "start_date": "2017-07-29", "end_date": "2021-02-08", "duration_months": 43, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}, {"company": "Acme Corp", "title": "Accountant", "start_date": "2014-05-16", "end_date": "2017-07-29", "duration_months": 39, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "M.Tech", "field_of_study": "Data Science", "start_year": 2017, "end_year": 2022, "grade": "7.58 CGPA", "tier": "tier_4"}], "skills": [{"name": "Next.js", "proficiency": "beginner", "endorsements": 9, "duration_months": 18}, {"name": "Java", "proficiency": "intermediate", "endorsements": 13, "duration_months": 15}, {"name": "Apache Flink", "proficiency": "intermediate", "endorsements": 4, "duration_months": 16}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 4, "duration_months": 8}, {"name": "Tally", "proficiency": "intermediate", "endorsements": 14, "duration_months": 12}, {"name": "PostgreSQL", "proficiency": "intermediate", "endorsements": 11, "duration_months": 25}, {"name": "REST APIs", "proficiency": "beginner", "endorsements": 15, "duration_months": 4}, {"name": "Hadoop", "proficiency": "intermediate", "endorsements": 4, "duration_months": 35}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2018}, {"name": "Scrum Master Certified", "issuer": "Scrum Alliance", "year": 2022}], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 38.7, "signup_date": "2025-08-11", "last_active_date": "2025-11-04", "open_to_work_flag": false, "profile_views_received_30d": 3, "applications_submitted_30d": 4, "recruiter_response_rate": 0.27, "avg_response_time_hours": 197.4, "skill_assessment_scores": {}, "connection_count": 35, "endorsements_received": 23, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 13.8, "max": 8.4}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 110, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.32, "offer_acceptance_rate": 0.17, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000018", "profile": {"anonymized_name": "Vivaan Reddy", "headline": "Frontend Engineer | Full-stack development", "summary": "Software engineer with 6.6 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Bhubaneswar, Odisha", "country": "India", "years_of_experience": 6.6, "current_title": "Frontend Engineer", "current_company": "Acme Corp", "current_company_size": "201-500", "current_industry": "Manufacturing"}, "career_history": [{"company": "Acme Corp", "title": "Frontend Engineer", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "Manufacturing", "company_size": "201-500", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Pied Piper", "title": "Frontend Engineer", "start_date": "2021-01-23", "end_date": "2023-09-10", "duration_months": 32, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Initech", "title": "Full Stack Developer", "start_date": "2019-12-30", "end_date": "2021-01-23", "duration_months": 13, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}], "education": [{"institution": "Lovely Professional University", "degree": "Ph.D", "field_of_study": "Computer Engineering", "start_year": 2016, "end_year": 2020, "grade": "7.25 CGPA", "tier": "tier_3"}], "skills": [{"name": "CNN", "proficiency": "advanced", "endorsements": 53, "duration_months": 55}, {"name": "Java", "proficiency": "intermediate", "endorsements": 10, "duration_months": 12}, {"name": "Accounting", "proficiency": "intermediate", "endorsements": 9, "duration_months": 20}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 3, "duration_months": 13}, {"name": "Node.js", "proficiency": "intermediate", "endorsements": 0, "duration_months": 9}, {"name": "Tailwind", "proficiency": "beginner", "endorsements": 10, "duration_months": 10}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 34.8, "signup_date": "2025-07-09", "last_active_date": "2026-02-18", "open_to_work_flag": false, "profile_views_received_30d": 88, "applications_submitted_30d": 11, "recruiter_response_rate": 0.16, "avg_response_time_hours": 154.6, "skill_assessment_scores": {}, "connection_count": 284, "endorsements_received": 49, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 12.3, "max": 26.4}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 41, "saved_by_recruiters_30d": 16, "interview_completion_rate": 0.7, "offer_acceptance_rate": 0.46, "verified_email": false, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000019", "profile": {"anonymized_name": "Myra Mishra", "headline": "Project Manager | 6.5+ yrs experience", "summary": "Professional with 6.5+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Trivandrum, Kerala", "country": "India", "years_of_experience": 6.5, "current_title": "Project Manager", "current_company": "Wayne Enterprises", "current_company_size": "10001+", "current_industry": "Conglomerate"}, "career_history": [{"company": "Wayne Enterprises", "title": "Project Manager", "start_date": "2022-10-15", "end_date": null, "duration_months": 44, "is_current": true, "industry": "Conglomerate", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Wayne Enterprises", "title": "Marketing Manager", "start_date": "2020-08-19", "end_date": "2022-10-08", "duration_months": 26, "is_current": false, "industry": "Conglomerate", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Pied Piper", "title": "Business Analyst", "start_date": "2020-01-15", "end_date": "2020-08-12", "duration_months": 7, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "IISc Bangalore", "degree": "M.Tech", "field_of_study": "Computer Science", "start_year": 2010, "end_year": 2014, "grade": "72%", "tier": "tier_1"}, {"institution": "IIT Guwahati", "degree": "M.Tech", "field_of_study": "Machine Learning", "start_year": 2002, "end_year": 2006, "grade": "7.34 CGPA", "tier": "tier_1"}], "skills": [{"name": "Figma", "proficiency": "intermediate", "endorsements": 13, "duration_months": 34}, {"name": "GraphQL", "proficiency": "beginner", "endorsements": 4, "duration_months": 12}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 4, "duration_months": 10}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 15, "duration_months": 2}, {"name": "YOLO", "proficiency": "intermediate", "endorsements": 10, "duration_months": 34}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 12, "duration_months": 9}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 11, "duration_months": 28}, {"name": "Azure", "proficiency": "intermediate", "endorsements": 6, "duration_months": 21}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 38.6, "signup_date": "2025-07-20", "last_active_date": "2026-05-21", "open_to_work_flag": false, "profile_views_received_30d": 61, "applications_submitted_30d": 9, "recruiter_response_rate": 0.34, "avg_response_time_hours": 100.0, "skill_assessment_scores": {}, "connection_count": 593, "endorsements_received": 25, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 12.5, "max": 7.7}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 141, "saved_by_recruiters_30d": 0, "interview_completion_rate": 0.31, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000020", "profile": {"anonymized_name": "Aditya Iyengar", "headline": "Mechanical Engineer | 6.3+ yrs experience", "summary": "Professional with 6.3+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Ahmedabad, Gujarat", "country": "India", "years_of_experience": 6.3, "current_title": "Mechanical Engineer", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Mechanical Engineer", "start_date": "2023-06-12", "end_date": null, "duration_months": 36, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Stark Industries", "title": "Graphic Designer", "start_date": "2020-07-27", "end_date": "2023-04-13", "duration_months": 33, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Dunder Mifflin", "title": "Civil Engineer", "start_date": "2020-01-29", "end_date": "2020-07-27", "duration_months": 6, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}], "education": [{"institution": "IIT Kharagpur", "degree": "B.E.", "field_of_study": "Computer Science", "start_year": 2018, "end_year": 2022, "grade": "6.67 CGPA", "tier": "tier_1"}], "skills": [{"name": "GraphQL", "proficiency": "beginner", "endorsements": 15, "duration_months": 18}, {"name": "TypeScript", "proficiency": "beginner", "endorsements": 7, "duration_months": 8}, {"name": "Flask", "proficiency": "beginner", "endorsements": 11, "duration_months": 16}, {"name": "Weights & Biases", "proficiency": "advanced", "endorsements": 50, "duration_months": 30}, {"name": "GCP", "proficiency": "intermediate", "endorsements": 2, "duration_months": 34}, {"name": "Salesforce CRM", "proficiency": "beginner", "endorsements": 8, "duration_months": 16}, {"name": "HTML", "proficiency": "intermediate", "endorsements": 2, "duration_months": 35}], "certifications": [{"name": "Scrum Master Certified", "issuer": "Scrum Alliance", "year": 2021}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 73.0, "signup_date": "2023-01-26", "last_active_date": "2025-10-05", "open_to_work_flag": false, "profile_views_received_30d": 38, "applications_submitted_30d": 9, "recruiter_response_rate": 0.55, "avg_response_time_hours": 207.8, "skill_assessment_scores": {"Weights & Biases": 53.7}, "connection_count": 479, "endorsements_received": 35, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 17.2, "max": 18.2}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 31, "saved_by_recruiters_30d": 11, "interview_completion_rate": 0.71, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0000021", "profile": {"anonymized_name": "Rahul Joshi", "headline": "Project Manager | AI enthusiast | Building with LLMs", "summary": "Project Manager with 14.5+ years of experience driving outcomes in my domain. I have built strong functional expertise in the typical responsibilities of the role, including team management, stakeholder communication, and project delivery. Recently I've been excited about how AI and GenAI tools can augment my work. I've been taking online courses on RAG and vector databases, experimenting with LangChain and the OpenAI API for side projects, and exploring how LLMs can streamline workflows in my current function. Open to roles that combine my existing domain experience with emerging AI technologies \u2014 I think the most interesting opportunities are at this intersection. Looking for positions where I can contribute both my functional expertise and grow my AI capabilities.", "location": "Bhubaneswar, Odisha", "country": "India", "years_of_experience": 14.5, "current_title": "Project Manager", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Project Manager", "start_date": "2023-12-09", "end_date": null, "duration_months": 30, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}, {"company": "Infosys", "title": "Marketing Manager", "start_date": "2021-02-22", "end_date": "2023-10-10", "duration_months": 32, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Stark Industries", "title": "Sales Executive", "start_date": "2019-08-25", "end_date": "2021-02-15", "duration_months": 18, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Dunder Mifflin", "title": "Customer Support", "start_date": "2015-06-17", "end_date": "2019-08-25", "duration_months": 51, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Wipro", "title": "Project Manager", "start_date": "2014-03-24", "end_date": "2015-06-17", "duration_months": 15, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "TCS", "title": "Customer Support", "start_date": "2011-12-05", "end_date": "2014-01-23", "duration_months": 26, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "B.Tech", "field_of_study": "Artificial Intelligence", "start_year": 2008, "end_year": 2011, "grade": "9.30 CGPA", "tier": "tier_4"}], "skills": [{"name": "Hadoop", "proficiency": "beginner", "endorsements": 10, "duration_months": 5}, {"name": "PostgreSQL", "proficiency": "beginner", "endorsements": 10, "duration_months": 4}, {"name": "Kafka", "proficiency": "beginner", "endorsements": 6, "duration_months": 6}, {"name": "Microservices", "proficiency": "intermediate", "endorsements": 0, "duration_months": 14}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 11, "duration_months": 26}, {"name": "TypeScript", "proficiency": "beginner", "endorsements": 6, "duration_months": 6}, {"name": "ETL", "proficiency": "beginner", "endorsements": 11, "duration_months": 3}, {"name": "Spring Boot", "proficiency": "beginner", "endorsements": 1, "duration_months": 12}, {"name": "Recommendation Systems", "proficiency": "advanced", "endorsements": 3, "duration_months": 13}, {"name": "Fine-tuning LLMs", "proficiency": "advanced", "endorsements": 4, "duration_months": 4}, {"name": "Prompt Engineering", "proficiency": "advanced", "endorsements": 4, "duration_months": 5}, {"name": "LangChain", "proficiency": "intermediate", "endorsements": 1, "duration_months": 7}, {"name": "Pinecone", "proficiency": "intermediate", "endorsements": 4, "duration_months": 16}, {"name": "Vector Search", "proficiency": "intermediate", "endorsements": 3, "duration_months": 13}, {"name": "Embeddings", "proficiency": "advanced", "endorsements": 4, "duration_months": 18}, {"name": "FAISS", "proficiency": "intermediate", "endorsements": 2, "duration_months": 8}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 58.5, "signup_date": "2026-02-10", "last_active_date": "2025-11-21", "open_to_work_flag": false, "profile_views_received_30d": 1, "applications_submitted_30d": 8, "recruiter_response_rate": 0.49, "avg_response_time_hours": 98.7, "skill_assessment_scores": {}, "connection_count": 52, "endorsements_received": 3, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 10.9, "max": 24.4}, "preferred_work_mode": "hybrid", "willing_to_relocate": true, "github_activity_score": 6.4, "search_appearance_30d": 8, "saved_by_recruiters_30d": 3, "interview_completion_rate": 0.53, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000022", "profile": {"anonymized_name": "Shaurya Chatterjee", "headline": "Mechanical Engineer | Driving business outcomes", "summary": "Professional with 1.1+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Sydney", "country": "Australia", "years_of_experience": 1.1, "current_title": "Mechanical Engineer", "current_company": "Hooli", "current_company_size": "1001-5000", "current_industry": "Software"}, "career_history": [{"company": "Hooli", "title": "Mechanical Engineer", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "Software", "company_size": "1001-5000", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}], "education": [{"institution": "VJTI Mumbai", "degree": "M.E.", "field_of_study": "Information Technology", "start_year": 2002, "end_year": 2006, "grade": "8.23 CGPA", "tier": "tier_2"}], "skills": [{"name": "OpenCV", "proficiency": "intermediate", "endorsements": 2, "duration_months": 11}, {"name": "Django", "proficiency": "intermediate", "endorsements": 0, "duration_months": 23}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 11, "duration_months": 14}, {"name": "Scrum", "proficiency": "intermediate", "endorsements": 7, "duration_months": 18}, {"name": "SQL", "proficiency": "intermediate", "endorsements": 3, "duration_months": 36}, {"name": "Java", "proficiency": "beginner", "endorsements": 11, "duration_months": 15}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 5, "duration_months": 29}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 1, "duration_months": 4}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2024}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 63.1, "signup_date": "2022-12-26", "last_active_date": "2026-05-03", "open_to_work_flag": true, "profile_views_received_30d": 39, "applications_submitted_30d": 2, "recruiter_response_rate": 0.27, "avg_response_time_hours": 30.2, "skill_assessment_scores": {}, "connection_count": 538, "endorsements_received": 21, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 12.3, "max": 8.5}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 136, "saved_by_recruiters_30d": 5, "interview_completion_rate": 0.45, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000023", "profile": {"anonymized_name": "Kavya Nair", "headline": "Software Engineer | Cloud & DevOps", "summary": "Software engineer with 3.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering \u2014 APIs, databases, queues, deployments. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "New York", "country": "USA", "years_of_experience": 3.7, "current_title": "Software Engineer", "current_company": "Acme Corp", "current_company_size": "201-500", "current_industry": "Manufacturing"}, "career_history": [{"company": "Acme Corp", "title": "Software Engineer", "start_date": "2025-04-02", "end_date": null, "duration_months": 14, "is_current": true, "industry": "Manufacturing", "company_size": "201-500", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Flipkart", "title": "Frontend Engineer", "start_date": "2022-10-15", "end_date": "2025-04-02", "duration_months": 30, "is_current": false, "industry": "E-commerce", "company_size": "10001+", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}], "education": [{"institution": "VJTI Mumbai", "degree": "B.E.", "field_of_study": "Data Science", "start_year": 2009, "end_year": 2013, "grade": "9.43 CGPA", "tier": "tier_2"}], "skills": [{"name": "BigQuery", "proficiency": "beginner", "endorsements": 8, "duration_months": 6}, {"name": "Marketing", "proficiency": "intermediate", "endorsements": 0, "duration_months": 15}, {"name": "Node.js", "proficiency": "intermediate", "endorsements": 3, "duration_months": 28}, {"name": "Django", "proficiency": "beginner", "endorsements": 13, "duration_months": 16}, {"name": "Salesforce CRM", "proficiency": "beginner", "endorsements": 9, "duration_months": 18}, {"name": "MongoDB", "proficiency": "beginner", "endorsements": 8, "duration_months": 10}, {"name": "ETL", "proficiency": "beginner", "endorsements": 12, "duration_months": 16}, {"name": "Redis", "proficiency": "beginner", "endorsements": 4, "duration_months": 10}, {"name": "Illustrator", "proficiency": "beginner", "endorsements": 1, "duration_months": 2}, {"name": "Rust", "proficiency": "intermediate", "endorsements": 2, "duration_months": 16}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 50.7, "signup_date": "2025-09-13", "last_active_date": "2026-04-06", "open_to_work_flag": false, "profile_views_received_30d": 10, "applications_submitted_30d": 2, "recruiter_response_rate": 0.57, "avg_response_time_hours": 64.8, "skill_assessment_scores": {}, "connection_count": 763, "endorsements_received": 39, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 17.4, "max": 20.5}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": 48.5, "search_appearance_30d": 239, "saved_by_recruiters_30d": 14, "interview_completion_rate": 0.9, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000024", "profile": {"anonymized_name": "Rajesh Arora", "headline": "HR Manager | 7.5+ yrs experience", "summary": "Professional with 7.5+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Trivandrum, Kerala", "country": "India", "years_of_experience": 7.5, "current_title": "HR Manager", "current_company": "TCS", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "TCS", "title": "HR Manager", "start_date": "2023-04-13", "end_date": null, "duration_months": 38, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Infosys", "title": "Accountant", "start_date": "2018-12-05", "end_date": "2023-02-12", "duration_months": 51, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}], "education": [{"institution": "Symbiosis International", "degree": "Ph.D", "field_of_study": "Computer Science", "start_year": 2008, "end_year": 2013, "grade": "7.65 CGPA", "tier": "tier_3"}], "skills": [{"name": "Figma", "proficiency": "beginner", "endorsements": 14, "duration_months": 15}, {"name": "Kubernetes", "proficiency": "beginner", "endorsements": 8, "duration_months": 17}, {"name": "Forecasting", "proficiency": "advanced", "endorsements": 43, "duration_months": 30}, {"name": "ETL", "proficiency": "intermediate", "endorsements": 11, "duration_months": 12}, {"name": "Node.js", "proficiency": "intermediate", "endorsements": 3, "duration_months": 15}, {"name": "Docker", "proficiency": "beginner", "endorsements": 5, "duration_months": 5}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 63.7, "signup_date": "2022-08-30", "last_active_date": "2026-01-20", "open_to_work_flag": false, "profile_views_received_30d": 71, "applications_submitted_30d": 4, "recruiter_response_rate": 0.78, "avg_response_time_hours": 238.2, "skill_assessment_scores": {"Forecasting": 65.1}, "connection_count": 445, "endorsements_received": 41, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 9.9, "max": 22.1}, "preferred_work_mode": "flexible", "willing_to_relocate": true, "github_activity_score": 46.3, "search_appearance_30d": 84, "saved_by_recruiters_30d": 7, "interview_completion_rate": 0.72, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000025", "profile": {"anonymized_name": "Anika Kumar", "headline": "Frontend Engineer | Cloud & DevOps", "summary": "Software engineer with 7.3 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Vizag, Andhra Pradesh", "country": "India", "years_of_experience": 7.3, "current_title": "Frontend Engineer", "current_company": "Tech Mahindra", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Tech Mahindra", "title": "Frontend Engineer", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Mindtree", "title": "Frontend Engineer", "start_date": "2019-09-17", "end_date": "2023-08-27", "duration_months": 48, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Zomato", "title": "Frontend Engineer", "start_date": "2019-03-21", "end_date": "2019-09-17", "duration_months": 6, "is_current": false, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."}], "education": [{"institution": "Regional Technical Institute", "degree": "Ph.D", "field_of_study": "Mechanical Engineering", "start_year": 2006, "end_year": 2010, "grade": "8.18 CGPA", "tier": "tier_4"}], "skills": [{"name": "JavaScript", "proficiency": "intermediate", "endorsements": 10, "duration_months": 26}, {"name": "Spark", "proficiency": "intermediate", "endorsements": 0, "duration_months": 22}, {"name": "GCP", "proficiency": "beginner", "endorsements": 7, "duration_months": 13}, {"name": "TypeScript", "proficiency": "beginner", "endorsements": 2, "duration_months": 17}, {"name": "LangChain", "proficiency": "advanced", "endorsements": 15, "duration_months": 34}, {"name": "Apache Flink", "proficiency": "intermediate", "endorsements": 2, "duration_months": 19}, {"name": "ETL", "proficiency": "beginner", "endorsements": 1, "duration_months": 18}, {"name": "Redis", "proficiency": "beginner", "endorsements": 0, "duration_months": 10}, {"name": "Data Pipelines", "proficiency": "intermediate", "endorsements": 9, "duration_months": 32}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2018}, {"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2025}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 70.7, "signup_date": "2023-12-18", "last_active_date": "2026-03-30", "open_to_work_flag": true, "profile_views_received_30d": 107, "applications_submitted_30d": 11, "recruiter_response_rate": 0.74, "avg_response_time_hours": 128.0, "skill_assessment_scores": {"LangChain": 40.0}, "connection_count": 276, "endorsements_received": 52, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 18.8, "max": 30.7}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 74, "saved_by_recruiters_30d": 9, "interview_completion_rate": 0.7, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000026", "profile": {"anonymized_name": "Neha Rao", "headline": "Graphic Designer | 6.8+ yrs experience", "summary": "Professional with 6.8+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Kochi, Kerala", "country": "India", "years_of_experience": 6.8, "current_title": "Graphic Designer", "current_company": "Initech", "current_company_size": "51-200", "current_industry": "Software"}, "career_history": [{"company": "Initech", "title": "Graphic Designer", "start_date": "2022-11-14", "end_date": null, "duration_months": 43, "is_current": true, "industry": "Software", "company_size": "51-200", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Acme Corp", "title": "Accountant", "start_date": "2021-03-24", "end_date": "2022-11-14", "duration_months": 20, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}, {"company": "Acme Corp", "title": "Project Manager", "start_date": "2019-10-31", "end_date": "2021-03-24", "duration_months": 17, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}], "education": [{"institution": "Generic State University", "degree": "M.Sc", "field_of_study": "Statistics", "start_year": 2008, "end_year": 2012, "grade": "81%", "tier": "tier_4"}, {"institution": "Lovely Professional University", "degree": "B.E.", "field_of_study": "Machine Learning", "start_year": 2008, "end_year": 2013, "grade": "83%", "tier": "tier_3"}], "skills": [{"name": "Apache Beam", "proficiency": "intermediate", "endorsements": 4, "duration_months": 18}, {"name": "Kubeflow", "proficiency": "intermediate", "endorsements": 14, "duration_months": 27}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 12, "duration_months": 8}, {"name": "ETL", "proficiency": "beginner", "endorsements": 15, "duration_months": 17}, {"name": "Django", "proficiency": "beginner", "endorsements": 6, "duration_months": 11}, {"name": "Docker", "proficiency": "beginner", "endorsements": 4, "duration_months": 9}, {"name": "Airflow", "proficiency": "intermediate", "endorsements": 7, "duration_months": 21}, {"name": "Kubernetes", "proficiency": "intermediate", "endorsements": 13, "duration_months": 22}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 80.3, "signup_date": "2023-12-08", "last_active_date": "2025-10-03", "open_to_work_flag": false, "profile_views_received_30d": 75, "applications_submitted_30d": 7, "recruiter_response_rate": 0.59, "avg_response_time_hours": 45.4, "skill_assessment_scores": {}, "connection_count": 321, "endorsements_received": 8, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 17.1, "max": 8.0}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 154, "saved_by_recruiters_30d": 11, "interview_completion_rate": 0.49, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0000027", "profile": {"anonymized_name": "Avni Pandey", "headline": "DevOps Engineer | Cloud & DevOps", "summary": "Software engineer with 3.9 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Kolkata, West Bengal", "country": "India", "years_of_experience": 3.9, "current_title": "DevOps Engineer", "current_company": "Infosys", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Infosys", "title": "DevOps Engineer", "start_date": "2023-06-12", "end_date": null, "duration_months": 36, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}, {"company": "Wipro", "title": "DevOps Engineer", "start_date": "2022-08-02", "end_date": "2023-05-29", "duration_months": 10, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."}], "education": [{"institution": "IIT Bombay", "degree": "Ph.D", "field_of_study": "Information Technology", "start_year": 2006, "end_year": 2010, "grade": "6.55 CGPA", "tier": "tier_1"}, {"institution": "VJTI Mumbai", "degree": "B.E.", "field_of_study": "Artificial Intelligence", "start_year": 2017, "end_year": 2020, "grade": "9.11 CGPA", "tier": "tier_2"}], "skills": [{"name": "Docker", "proficiency": "intermediate", "endorsements": 1, "duration_months": 9}, {"name": "YOLO", "proficiency": "advanced", "endorsements": 31, "duration_months": 20}, {"name": "PEFT", "proficiency": "advanced", "endorsements": 39, "duration_months": 35}, {"name": "Webpack", "proficiency": "intermediate", "endorsements": 0, "duration_months": 33}, {"name": "Data Science", "proficiency": "advanced", "endorsements": 18, "duration_months": 24}, {"name": "AWS", "proficiency": "beginner", "endorsements": 4, "duration_months": 16}, {"name": "Java", "proficiency": "intermediate", "endorsements": 2, "duration_months": 15}, {"name": "Angular", "proficiency": "intermediate", "endorsements": 4, "duration_months": 25}, {"name": "Databricks", "proficiency": "intermediate", "endorsements": 3, "duration_months": 31}, {"name": "ETL", "proficiency": "intermediate", "endorsements": 5, "duration_months": 11}, {"name": "Marketing", "proficiency": "beginner", "endorsements": 15, "duration_months": 14}, {"name": "Information Retrieval", "proficiency": "intermediate", "endorsements": 5, "duration_months": 32}, {"name": "Weights & Biases", "proficiency": "advanced", "endorsements": 31, "duration_months": 44}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 12, "duration_months": 19}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 8, "duration_months": 9}, {"name": "Illustrator", "proficiency": "beginner", "endorsements": 1, "duration_months": 11}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 31.0, "signup_date": "2023-03-07", "last_active_date": "2026-05-07", "open_to_work_flag": true, "profile_views_received_30d": 89, "applications_submitted_30d": 5, "recruiter_response_rate": 0.58, "avg_response_time_hours": 112.3, "skill_assessment_scores": {"YOLO": 60.2, "PEFT": 50.5, "Data Science": 35.1}, "connection_count": 282, "endorsements_received": 24, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 17.9, "max": 20.2}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": 38.6, "search_appearance_30d": 136, "saved_by_recruiters_30d": 6, "interview_completion_rate": 0.61, "offer_acceptance_rate": 0.65, "verified_email": false, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000028", "profile": {"anonymized_name": "Rohan Krishnan", "headline": "Operations Manager | Driving business outcomes", "summary": "Professional with 1.1+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Dubai", "country": "UAE", "years_of_experience": 1.1, "current_title": "Operations Manager", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Operations Manager", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "Symbiosis International", "degree": "M.Tech", "field_of_study": "Mathematics", "start_year": 2014, "end_year": 2017, "grade": "7.85 CGPA", "tier": "tier_3"}], "skills": [{"name": "Snowflake", "proficiency": "beginner", "endorsements": 6, "duration_months": 3}, {"name": "React", "proficiency": "beginner", "endorsements": 11, "duration_months": 7}, {"name": "JavaScript", "proficiency": "beginner", "endorsements": 6, "duration_months": 10}, {"name": "Tailwind", "proficiency": "beginner", "endorsements": 13, "duration_months": 15}, {"name": "REST APIs", "proficiency": "intermediate", "endorsements": 6, "duration_months": 21}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 9, "duration_months": 30}, {"name": "Data Pipelines", "proficiency": "intermediate", "endorsements": 4, "duration_months": 23}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 9, "duration_months": 18}, {"name": "CNN", "proficiency": "intermediate", "endorsements": 8, "duration_months": 29}, {"name": "Content Writing", "proficiency": "beginner", "endorsements": 9, "duration_months": 18}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2020}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 51.2, "signup_date": "2025-09-23", "last_active_date": "2026-03-31", "open_to_work_flag": false, "profile_views_received_30d": 6, "applications_submitted_30d": 7, "recruiter_response_rate": 0.14, "avg_response_time_hours": 13.2, "skill_assessment_scores": {}, "connection_count": 524, "endorsements_received": 1, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 12.9, "max": 17.2}, "preferred_work_mode": "remote", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 68, "saved_by_recruiters_30d": 4, "interview_completion_rate": 0.86, "offer_acceptance_rate": 0.49, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000029", "profile": {"anonymized_name": "Priya Sethi", "headline": "Business Analyst | Driving business outcomes", "summary": "Professional with 7.2+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Noida, Uttar Pradesh", "country": "India", "years_of_experience": 7.2, "current_title": "Business Analyst", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Business Analyst", "start_date": "2025-02-01", "end_date": null, "duration_months": 16, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Globex Inc", "title": "Mechanical Engineer", "start_date": "2023-12-09", "end_date": "2025-02-01", "duration_months": 14, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "TCS", "title": "Civil Engineer", "start_date": "2019-05-27", "end_date": "2023-12-02", "duration_months": 55, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}], "education": [{"institution": "Symbiosis International", "degree": "B.Tech", "field_of_study": "Artificial Intelligence", "start_year": 2007, "end_year": 2011, "grade": "6.59 CGPA", "tier": "tier_3"}], "skills": [{"name": "Node.js", "proficiency": "beginner", "endorsements": 9, "duration_months": 18}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 2, "duration_months": 5}, {"name": "Tailwind", "proficiency": "intermediate", "endorsements": 5, "duration_months": 21}, {"name": "Hadoop", "proficiency": "beginner", "endorsements": 10, "duration_months": 4}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 1, "duration_months": 10}, {"name": "CI/CD", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 15, "duration_months": 17}, {"name": "Terraform", "proficiency": "beginner", "endorsements": 9, "duration_months": 10}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 40.5, "signup_date": "2025-06-17", "last_active_date": "2025-09-29", "open_to_work_flag": false, "profile_views_received_30d": 51, "applications_submitted_30d": 8, "recruiter_response_rate": 0.12, "avg_response_time_hours": 48.4, "skill_assessment_scores": {}, "connection_count": 297, "endorsements_received": 4, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 17.5, "max": 19.1}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": 42.5, "search_appearance_30d": 150, "saved_by_recruiters_30d": 8, "interview_completion_rate": 0.67, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0000030", "profile": {"anonymized_name": "Ritu Pillai", "headline": "Marketing Manager | Driving business outcomes", "summary": "Professional with 10.0+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Kochi, Kerala", "country": "India", "years_of_experience": 10.0, "current_title": "Marketing Manager", "current_company": "Dunder Mifflin", "current_company_size": "201-500", "current_industry": "Paper Products"}, "career_history": [{"company": "Dunder Mifflin", "title": "Marketing Manager", "start_date": "2022-03-19", "end_date": null, "duration_months": 51, "is_current": true, "industry": "Paper Products", "company_size": "201-500", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Hooli", "title": "Sales Executive", "start_date": "2018-07-08", "end_date": "2022-03-19", "duration_months": 45, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}, {"company": "Hooli", "title": "Content Writer", "start_date": "2016-08-17", "end_date": "2018-06-08", "duration_months": 22, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}], "education": [{"institution": "Generic State University", "degree": "B.E.", "field_of_study": "Computer Engineering", "start_year": 2010, "end_year": 2014, "grade": "81%", "tier": "tier_4"}, {"institution": "Generic State University", "degree": "Ph.D", "field_of_study": "Artificial Intelligence", "start_year": 2017, "end_year": 2020, "grade": "7.98 CGPA", "tier": "tier_4"}], "skills": [{"name": "gRPC", "proficiency": "intermediate", "endorsements": 3, "duration_months": 36}, {"name": "Apache Beam", "proficiency": "beginner", "endorsements": 5, "duration_months": 6}, {"name": "GraphQL", "proficiency": "intermediate", "endorsements": 2, "duration_months": 22}, {"name": "Java", "proficiency": "intermediate", "endorsements": 14, "duration_months": 11}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 2, "duration_months": 22}, {"name": "Microservices", "proficiency": "beginner", "endorsements": 11, "duration_months": 5}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 8, "duration_months": 8}, {"name": "Accounting", "proficiency": "intermediate", "endorsements": 3, "duration_months": 30}, {"name": "HTML", "proficiency": "intermediate", "endorsements": 12, "duration_months": 9}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 59.7, "signup_date": "2025-09-25", "last_active_date": "2025-10-27", "open_to_work_flag": false, "profile_views_received_30d": 58, "applications_submitted_30d": 0, "recruiter_response_rate": 0.66, "avg_response_time_hours": 131.1, "skill_assessment_scores": {}, "connection_count": 552, "endorsements_received": 45, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 14.7, "max": 14.2}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": 21.7, "search_appearance_30d": 54, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.73, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000031", "profile": {"anonymized_name": "Ela Singh", "headline": "Recommendation Systems Engineer | Search, Ranking & Retrieval", "summary": "Machine learning engineer with 6.0 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I've learned that most retrieval problems are actually evaluation problems in disguise. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I'd own a meaningful piece of the ML stack.", "location": "Hyderabad, Telangana", "country": "India", "years_of_experience": 6.0, "current_title": "Recommendation Systems Engineer", "current_company": "Swiggy", "current_company_size": "5001-10000", "current_industry": "Food Delivery"}, "career_history": [{"company": "Swiggy", "title": "Recommendation Systems Engineer", "start_date": "2025-04-02", "end_date": null, "duration_months": 14, "is_current": true, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \u2014 that work was as important as the modeling itself."}, {"company": "Mad Street Den", "title": "Search Engineer", "start_date": "2023-10-10", "end_date": "2025-02-01", "duration_months": 16, "is_current": false, "industry": "AI/ML", "company_size": "201-500", "description": "Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \u2014 that work was as important as the modeling itself."}, {"company": "Uber", "title": "NLP Engineer", "start_date": "2021-07-22", "end_date": "2023-10-10", "duration_months": 27, "is_current": false, "industry": "Transportation", "company_size": "10001+", "description": "Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \u2014 that work was as important as the modeling itself."}, {"company": "Zomato", "title": "Applied ML Engineer", "start_date": "2020-06-27", "end_date": "2021-07-22", "duration_months": 13, "is_current": false, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Owned the ranking layer for an e-commerce search product, evolving it from a hand-tuned scoring function to a learning-to-rank model over 9 months. Designed the relevance labeling pipeline (mix of click-through data and explicit human judgments), the feature pipeline, and the training/eval workflow. Most of the work was infrastructure and data quality \u2014 the modeling part was almost the easy bit. Final model improved revenue-per-search by 12%."}], "education": [{"institution": "SRM University", "degree": "M.Tech", "field_of_study": "Computer Engineering", "start_year": 2002, "end_year": 2006, "grade": "9.16 CGPA", "tier": "tier_2"}], "skills": [{"name": "Go", "proficiency": "intermediate", "endorsements": 7, "duration_months": 19}, {"name": "MLflow", "proficiency": "advanced", "endorsements": 59, "duration_months": 21}, {"name": "FAISS", "proficiency": "advanced", "endorsements": 19, "duration_months": 35}, {"name": "Pinecone", "proficiency": "expert", "endorsements": 34, "duration_months": 88}, {"name": "Angular", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "Image Classification", "proficiency": "advanced", "endorsements": 56, "duration_months": 28}, {"name": "Machine Learning", "proficiency": "advanced", "endorsements": 43, "duration_months": 23}, {"name": "Speech Recognition", "proficiency": "intermediate", "endorsements": 14, "duration_months": 24}, {"name": "BentoML", "proficiency": "intermediate", "endorsements": 6, "duration_months": 14}, {"name": "MLOps", "proficiency": "intermediate", "endorsements": 15, "duration_months": 36}, {"name": "Embeddings", "proficiency": "expert", "endorsements": 48, "duration_months": 60}, {"name": "Information Retrieval", "proficiency": "expert", "endorsements": 2, "duration_months": 84}, {"name": "Hugging Face Transformers", "proficiency": "expert", "endorsements": 18, "duration_months": 36}, {"name": "Feature Engineering", "proficiency": "advanced", "endorsements": 38, "duration_months": 26}, {"name": "Sentence Transformers", "proficiency": "expert", "endorsements": 16, "duration_months": 69}, {"name": "scikit-learn", "proficiency": "advanced", "endorsements": 41, "duration_months": 60}, {"name": "Marketing", "proficiency": "intermediate", "endorsements": 11, "duration_months": 36}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 83.4, "signup_date": "2026-01-28", "last_active_date": "2026-05-24", "open_to_work_flag": true, "profile_views_received_30d": 194, "applications_submitted_30d": 2, "recruiter_response_rate": 0.91, "avg_response_time_hours": 76.1, "skill_assessment_scores": {"MLflow": 75.1, "FAISS": 68.4, "Pinecone": 53.6, "Image Classification": 57.1}, "connection_count": 832, "endorsements_received": 177, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 27.3, "max": 60.2}, "preferred_work_mode": "flexible", "willing_to_relocate": true, "github_activity_score": 32.6, "search_appearance_30d": 778, "saved_by_recruiters_30d": 13, "interview_completion_rate": 0.6, "offer_acceptance_rate": 0.38, "verified_email": false, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000032", "profile": {"anonymized_name": "Pranav Agarwal", "headline": ".NET Developer | Full-stack development", "summary": "Software engineer with 8.1 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering \u2014 APIs, databases, queues, deployments. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Gurgaon, Haryana", "country": "India", "years_of_experience": 8.1, "current_title": ".NET Developer", "current_company": "Cognizant", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Cognizant", "title": ".NET Developer", "start_date": "2024-02-07", "end_date": null, "duration_months": 28, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}, {"company": "HCL", "title": "Cloud Engineer", "start_date": "2021-11-19", "end_date": "2024-02-07", "duration_months": 27, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Globex Inc", "title": "Mobile Developer", "start_date": "2018-07-24", "end_date": "2021-11-05", "duration_months": 40, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."}], "education": [{"institution": "VIT Chennai", "degree": "M.Sc", "field_of_study": "Machine Learning", "start_year": 2017, "end_year": 2020, "grade": "8.37 CGPA", "tier": "tier_3"}, {"institution": "Amity University", "degree": "Ph.D", "field_of_study": "Physics", "start_year": 2011, "end_year": 2015, "grade": "7.95 CGPA", "tier": "tier_3"}], "skills": [{"name": "Speech Recognition", "proficiency": "advanced", "endorsements": 36, "duration_months": 19}, {"name": "Project Management", "proficiency": "beginner", "endorsements": 6, "duration_months": 17}, {"name": "REST APIs", "proficiency": "beginner", "endorsements": 13, "duration_months": 6}, {"name": "CSS", "proficiency": "intermediate", "endorsements": 15, "duration_months": 27}, {"name": "Embeddings", "proficiency": "advanced", "endorsements": 30, "duration_months": 30}, {"name": "Hadoop", "proficiency": "beginner", "endorsements": 0, "duration_months": 8}, {"name": "Spark", "proficiency": "intermediate", "endorsements": 14, "duration_months": 30}, {"name": "Python", "proficiency": "intermediate", "endorsements": 2, "duration_months": 13}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 6, "duration_months": 8}, {"name": "OpenCV", "proficiency": "advanced", "endorsements": 45, "duration_months": 54}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 35.4, "signup_date": "2023-12-20", "last_active_date": "2025-12-29", "open_to_work_flag": false, "profile_views_received_30d": 80, "applications_submitted_30d": 3, "recruiter_response_rate": 0.69, "avg_response_time_hours": 58.6, "skill_assessment_scores": {}, "connection_count": 404, "endorsements_received": 22, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 18.3, "max": 15.7}, "preferred_work_mode": "flexible", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 56, "saved_by_recruiters_30d": 4, "interview_completion_rate": 0.78, "offer_acceptance_rate": 0.25, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000033", "profile": {"anonymized_name": "Shreya Nair", "headline": "Graphic Designer | Helping teams scale", "summary": "Professional with 8.6+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Pune, Maharashtra", "country": "India", "years_of_experience": 8.6, "current_title": "Graphic Designer", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Graphic Designer", "start_date": "2023-11-09", "end_date": null, "duration_months": 31, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Dunder Mifflin", "title": "Project Manager", "start_date": "2020-07-20", "end_date": "2023-11-02", "duration_months": 40, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Acme Corp", "title": "Content Writer", "start_date": "2018-01-02", "end_date": "2020-07-20", "duration_months": 31, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "M.S.", "field_of_study": "Computer Science", "start_year": 2014, "end_year": 2017, "grade": "9.32 CGPA", "tier": "tier_4"}], "skills": [{"name": "Kubernetes", "proficiency": "beginner", "endorsements": 0, "duration_months": 9}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 4, "duration_months": 8}, {"name": "Snowflake", "proficiency": "intermediate", "endorsements": 11, "duration_months": 12}, {"name": "CI/CD", "proficiency": "intermediate", "endorsements": 11, "duration_months": 29}, {"name": "SEO", "proficiency": "intermediate", "endorsements": 9, "duration_months": 36}, {"name": "ETL", "proficiency": "intermediate", "endorsements": 0, "duration_months": 20}, {"name": "Airflow", "proficiency": "intermediate", "endorsements": 11, "duration_months": 16}, {"name": "TypeScript", "proficiency": "intermediate", "endorsements": 13, "duration_months": 15}, {"name": "Content Writing", "proficiency": "intermediate", "endorsements": 13, "duration_months": 11}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 5, "duration_months": 26}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2019}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 74.0, "signup_date": "2026-03-13", "last_active_date": "2026-03-27", "open_to_work_flag": true, "profile_views_received_30d": 42, "applications_submitted_30d": 9, "recruiter_response_rate": 0.08, "avg_response_time_hours": 210.9, "skill_assessment_scores": {}, "connection_count": 410, "endorsements_received": 29, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 8.3, "max": 13.0}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": 38.3, "search_appearance_30d": 98, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.37, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000034", "profile": {"anonymized_name": "Yash Chatterjee", "headline": "Business Analyst | Driving business outcomes", "summary": "Professional with 14.5+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Ahmedabad, Gujarat", "country": "India", "years_of_experience": 14.5, "current_title": "Business Analyst", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Business Analyst", "start_date": "2025-04-02", "end_date": null, "duration_months": 14, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}, {"company": "Hooli", "title": "Mechanical Engineer", "start_date": "2023-05-13", "end_date": "2025-02-01", "duration_months": 21, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Infosys", "title": "Business Analyst", "start_date": "2021-02-22", "end_date": "2023-04-13", "duration_months": 26, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "TCS", "title": "Accountant", "start_date": "2017-11-10", "end_date": "2021-02-22", "duration_months": 40, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Hooli", "title": "Accountant", "start_date": "2016-01-20", "end_date": "2017-11-10", "duration_months": 22, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Pied Piper", "title": "Content Writer", "start_date": "2012-12-06", "end_date": "2016-01-20", "duration_months": 38, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Stark Industries", "title": "Content Writer", "start_date": "2012-01-11", "end_date": "2012-10-07", "duration_months": 9, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "B.E.", "field_of_study": "Computer Engineering", "start_year": 2005, "end_year": 2010, "grade": "8.97 CGPA", "tier": "tier_4"}], "skills": [{"name": "GraphQL", "proficiency": "beginner", "endorsements": 2, "duration_months": 3}, {"name": "Excel", "proficiency": "intermediate", "endorsements": 15, "duration_months": 19}, {"name": "Node.js", "proficiency": "beginner", "endorsements": 6, "duration_months": 6}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 6, "duration_months": 13}, {"name": "Salesforce CRM", "proficiency": "intermediate", "endorsements": 6, "duration_months": 27}, {"name": "Flask", "proficiency": "intermediate", "endorsements": 8, "duration_months": 28}, {"name": "React", "proficiency": "beginner", "endorsements": 14, "duration_months": 2}, {"name": "Azure", "proficiency": "beginner", "endorsements": 1, "duration_months": 5}, {"name": "Redux", "proficiency": "beginner", "endorsements": 15, "duration_months": 3}, {"name": "Next.js", "proficiency": "intermediate", "endorsements": 7, "duration_months": 21}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 41.2, "signup_date": "2024-01-15", "last_active_date": "2026-01-03", "open_to_work_flag": false, "profile_views_received_30d": 25, "applications_submitted_30d": 7, "recruiter_response_rate": 0.15, "avg_response_time_hours": 253.5, "skill_assessment_scores": {}, "connection_count": 226, "endorsements_received": 27, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 14.4, "max": 28.0}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 143, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.41, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000035", "profile": {"anonymized_name": "Vikram Verma", "headline": "Full Stack Developer | Backend systems & APIs", "summary": "Software engineer with 4.3 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Hyderabad, Telangana", "country": "India", "years_of_experience": 4.3, "current_title": "Full Stack Developer", "current_company": "Globex Inc", "current_company_size": "501-1000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Globex Inc", "title": "Full Stack Developer", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "Manufacturing", "company_size": "501-1000", "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."}, {"company": "Wipro", "title": "Software Engineer", "start_date": "2022-03-19", "end_date": "2023-09-10", "duration_months": 18, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."}], "education": [{"institution": "Generic State University", "degree": "B.E.", "field_of_study": "Civil Engineering", "start_year": 2010, "end_year": 2013, "grade": "90%", "tier": "tier_4"}, {"institution": "Bharati Vidyapeeth", "degree": "M.S.", "field_of_study": "Data Science", "start_year": 2010, "end_year": 2015, "grade": "7.08 CGPA", "tier": "tier_3"}], "skills": [{"name": "Snowflake", "proficiency": "intermediate", "endorsements": 11, "duration_months": 27}, {"name": "BigQuery", "proficiency": "beginner", "endorsements": 15, "duration_months": 6}, {"name": "Recommendation Systems", "proficiency": "intermediate", "endorsements": 2, "duration_months": 34}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 14, "duration_months": 18}, {"name": "Docker", "proficiency": "beginner", "endorsements": 3, "duration_months": 11}, {"name": "MongoDB", "proficiency": "intermediate", "endorsements": 11, "duration_months": 16}, {"name": "PostgreSQL", "proficiency": "intermediate", "endorsements": 14, "duration_months": 13}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 0, "duration_months": 19}, {"name": "Kafka", "proficiency": "intermediate", "endorsements": 14, "duration_months": 29}, {"name": "Speech Recognition", "proficiency": "intermediate", "endorsements": 5, "duration_months": 33}, {"name": "BentoML", "proficiency": "advanced", "endorsements": 40, "duration_months": 59}, {"name": "Go", "proficiency": "beginner", "endorsements": 1, "duration_months": 11}, {"name": "Next.js", "proficiency": "intermediate", "endorsements": 15, "duration_months": 22}, {"name": "dbt", "proficiency": "beginner", "endorsements": 2, "duration_months": 9}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2022}, {"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2020}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 56.2, "signup_date": "2024-08-13", "last_active_date": "2026-02-06", "open_to_work_flag": false, "profile_views_received_30d": 7, "applications_submitted_30d": 4, "recruiter_response_rate": 0.34, "avg_response_time_hours": 178.0, "skill_assessment_scores": {}, "connection_count": 398, "endorsements_received": 45, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 11.2, "max": 22.8}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 173, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.5, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000036", "profile": {"anonymized_name": "Ananya Bose", "headline": "Project Manager | 11.3+ yrs experience", "summary": "Professional with 11.3+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Trivandrum, Kerala", "country": "India", "years_of_experience": 11.3, "current_title": "Project Manager", "current_company": "Initech", "current_company_size": "51-200", "current_industry": "Software"}, "career_history": [{"company": "Initech", "title": "Project Manager", "start_date": "2025-02-01", "end_date": null, "duration_months": 16, "is_current": true, "industry": "Software", "company_size": "51-200", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}, {"company": "Hooli", "title": "Content Writer", "start_date": "2023-08-11", "end_date": "2025-02-01", "duration_months": 18, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Dunder Mifflin", "title": "HR Manager", "start_date": "2019-11-30", "end_date": "2023-08-11", "duration_months": 45, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}, {"company": "TCS", "title": "Civil Engineer", "start_date": "2017-12-10", "end_date": "2019-11-30", "duration_months": 24, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Wayne Enterprises", "title": "Marketing Manager", "start_date": "2015-05-18", "end_date": "2017-12-03", "duration_months": 31, "is_current": false, "industry": "Conglomerate", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}], "education": [{"institution": "KIIT University", "degree": "M.S.", "field_of_study": "Commerce", "start_year": 2002, "end_year": 2006, "grade": "8.89 CGPA", "tier": "tier_3"}], "skills": [{"name": "Figma", "proficiency": "beginner", "endorsements": 6, "duration_months": 13}, {"name": "MongoDB", "proficiency": "beginner", "endorsements": 4, "duration_months": 8}, {"name": "PowerPoint", "proficiency": "beginner", "endorsements": 5, "duration_months": 3}, {"name": "CSS", "proficiency": "beginner", "endorsements": 5, "duration_months": 14}, {"name": "Excel", "proficiency": "intermediate", "endorsements": 9, "duration_months": 28}, {"name": "GraphQL", "proficiency": "beginner", "endorsements": 5, "duration_months": 18}, {"name": "Object Detection", "proficiency": "advanced", "endorsements": 39, "duration_months": 37}, {"name": "Vue.js", "proficiency": "beginner", "endorsements": 13, "duration_months": 4}, {"name": "Sales", "proficiency": "beginner", "endorsements": 4, "duration_months": 9}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 81.8, "signup_date": "2025-09-05", "last_active_date": "2025-12-12", "open_to_work_flag": true, "profile_views_received_30d": 70, "applications_submitted_30d": 4, "recruiter_response_rate": 0.4, "avg_response_time_hours": 236.6, "skill_assessment_scores": {}, "connection_count": 324, "endorsements_received": 3, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 15.3, "max": 9.7}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 175, "saved_by_recruiters_30d": 0, "interview_completion_rate": 0.78, "offer_acceptance_rate": 0.46, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000037", "profile": {"anonymized_name": "Ved Sen", "headline": "Business Analyst | 14.3+ yrs experience", "summary": "Professional with 14.3+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Dubai", "country": "UAE", "years_of_experience": 14.3, "current_title": "Business Analyst", "current_company": "Stark Industries", "current_company_size": "1001-5000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Stark Industries", "title": "Business Analyst", "start_date": "2022-03-19", "end_date": null, "duration_months": 51, "is_current": true, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Initech", "title": "Civil Engineer", "start_date": "2019-08-18", "end_date": "2022-03-05", "duration_months": 31, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}, {"company": "Stark Industries", "title": "Content Writer", "start_date": "2016-01-06", "end_date": "2019-08-18", "duration_months": 44, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}, {"company": "Hooli", "title": "Mechanical Engineer", "start_date": "2013-07-20", "end_date": "2016-01-06", "duration_months": 30, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Acme Corp", "title": "HR Manager", "start_date": "2012-04-26", "end_date": "2013-06-20", "duration_months": 14, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "B.Tech", "field_of_study": "Machine Learning", "start_year": 2001, "end_year": 2005, "grade": "69%", "tier": "tier_4"}, {"institution": "Lovely Professional University", "degree": "M.Tech", "field_of_study": "Statistics", "start_year": 2013, "end_year": 2016, "grade": "6.81 CGPA", "tier": "tier_3"}], "skills": [{"name": "Databricks", "proficiency": "intermediate", "endorsements": 12, "duration_months": 32}, {"name": "Docker", "proficiency": "intermediate", "endorsements": 6, "duration_months": 35}, {"name": "Flask", "proficiency": "beginner", "endorsements": 5, "duration_months": 14}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 2, "duration_months": 33}, {"name": "Terraform", "proficiency": "beginner", "endorsements": 3, "duration_months": 18}, {"name": "Tally", "proficiency": "intermediate", "endorsements": 14, "duration_months": 24}, {"name": "TTS", "proficiency": "intermediate", "endorsements": 3, "duration_months": 13}, {"name": "Apache Beam", "proficiency": "intermediate", "endorsements": 15, "duration_months": 23}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2024}], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 25.6, "signup_date": "2025-04-11", "last_active_date": "2025-12-11", "open_to_work_flag": false, "profile_views_received_30d": 80, "applications_submitted_30d": 6, "recruiter_response_rate": 0.78, "avg_response_time_hours": 107.1, "skill_assessment_scores": {}, "connection_count": 503, "endorsements_received": 13, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 8.8, "max": 14.2}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 178, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.82, "offer_acceptance_rate": 0.19, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000038", "profile": {"anonymized_name": "Myra Trivedi", "headline": "Java Developer | Cloud & DevOps", "summary": "Software engineer with 6.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Coimbatore, Tamil Nadu", "country": "India", "years_of_experience": 6.7, "current_title": "Java Developer", "current_company": "Swiggy", "current_company_size": "5001-10000", "current_industry": "Food Delivery"}, "career_history": [{"company": "Swiggy", "title": "Java Developer", "start_date": "2023-11-09", "end_date": null, "duration_months": 31, "is_current": true, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."}, {"company": "Globex Inc", "title": ".NET Developer", "start_date": "2022-09-15", "end_date": "2023-11-09", "duration_months": 14, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}, {"company": "Hooli", "title": "DevOps Engineer", "start_date": "2019-11-30", "end_date": "2022-09-15", "duration_months": 34, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."}], "education": [{"institution": "VIT Chennai", "degree": "B.Sc", "field_of_study": "Computer Engineering", "start_year": 2015, "end_year": 2020, "grade": "70%", "tier": "tier_3"}], "skills": [{"name": "Kubeflow", "proficiency": "intermediate", "endorsements": 3, "duration_months": 26}, {"name": "Django", "proficiency": "beginner", "endorsements": 2, "duration_months": 18}, {"name": "Redux", "proficiency": "intermediate", "endorsements": 3, "duration_months": 13}, {"name": "Weaviate", "proficiency": "advanced", "endorsements": 37, "duration_months": 27}, {"name": "PowerPoint", "proficiency": "beginner", "endorsements": 15, "duration_months": 9}, {"name": "Figma", "proficiency": "beginner", "endorsements": 9, "duration_months": 8}, {"name": "Docker", "proficiency": "beginner", "endorsements": 12, "duration_months": 3}, {"name": "GraphQL", "proficiency": "intermediate", "endorsements": 13, "duration_months": 27}, {"name": "Agile", "proficiency": "intermediate", "endorsements": 14, "duration_months": 24}, {"name": "MLOps", "proficiency": "intermediate", "endorsements": 13, "duration_months": 26}, {"name": "Azure", "proficiency": "intermediate", "endorsements": 14, "duration_months": 27}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 35.8, "signup_date": "2026-03-25", "last_active_date": "2026-03-31", "open_to_work_flag": true, "profile_views_received_30d": 102, "applications_submitted_30d": 9, "recruiter_response_rate": 0.2, "avg_response_time_hours": 61.0, "skill_assessment_scores": {}, "connection_count": 316, "endorsements_received": 51, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 9.2, "max": 15.9}, "preferred_work_mode": "flexible", "willing_to_relocate": true, "github_activity_score": 37.8, "search_appearance_30d": 300, "saved_by_recruiters_30d": 18, "interview_completion_rate": 0.75, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000039", "profile": {"anonymized_name": "Sai Saxena", "headline": "Marketing Manager | Helping teams scale", "summary": "Professional with 3.9+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Bhubaneswar, Odisha", "country": "India", "years_of_experience": 3.9, "current_title": "Marketing Manager", "current_company": "Acme Corp", "current_company_size": "201-500", "current_industry": "Manufacturing"}, "career_history": [{"company": "Acme Corp", "title": "Marketing Manager", "start_date": "2024-08-05", "end_date": null, "duration_months": 22, "is_current": true, "industry": "Manufacturing", "company_size": "201-500", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Stark Industries", "title": "Mechanical Engineer", "start_date": "2022-08-16", "end_date": "2024-08-05", "duration_months": 24, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "Chandigarh University", "degree": "B.Tech", "field_of_study": "Electrical Engineering", "start_year": 2009, "end_year": 2014, "grade": "66%", "tier": "tier_3"}, {"institution": "Chandigarh University", "degree": "M.E.", "field_of_study": "Artificial Intelligence", "start_year": 2007, "end_year": 2011, "grade": "82%", "tier": "tier_3"}], "skills": [{"name": "Spark", "proficiency": "intermediate", "endorsements": 9, "duration_months": 15}, {"name": "Tailwind", "proficiency": "intermediate", "endorsements": 9, "duration_months": 8}, {"name": "Sales", "proficiency": "beginner", "endorsements": 6, "duration_months": 8}, {"name": "CI/CD", "proficiency": "intermediate", "endorsements": 11, "duration_months": 35}, {"name": "Illustrator", "proficiency": "intermediate", "endorsements": 9, "duration_months": 21}, {"name": "Hadoop", "proficiency": "intermediate", "endorsements": 15, "duration_months": 26}, {"name": "Microservices", "proficiency": "intermediate", "endorsements": 13, "duration_months": 19}, {"name": "REST APIs", "proficiency": "beginner", "endorsements": 15, "duration_months": 12}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 7, "duration_months": 18}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 28.3, "signup_date": "2025-04-14", "last_active_date": "2026-04-18", "open_to_work_flag": false, "profile_views_received_30d": 22, "applications_submitted_30d": 0, "recruiter_response_rate": 0.52, "avg_response_time_hours": 40.1, "skill_assessment_scores": {}, "connection_count": 102, "endorsements_received": 9, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 17.4, "max": 16.0}, "preferred_work_mode": "hybrid", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 146, "saved_by_recruiters_30d": 6, "interview_completion_rate": 0.74, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000040", "profile": {"anonymized_name": "Dev Vora", "headline": "Customer Support | Helping teams scale", "summary": "Professional with 1.6+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Kochi, Kerala", "country": "India", "years_of_experience": 1.6, "current_title": "Customer Support", "current_company": "Globex Inc", "current_company_size": "501-1000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Globex Inc", "title": "Customer Support", "start_date": "2024-11-03", "end_date": null, "duration_months": 19, "is_current": true, "industry": "Manufacturing", "company_size": "501-1000", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}], "education": [{"institution": "Local Engineering College", "degree": "B.Tech", "field_of_study": "MBA", "start_year": 2010, "end_year": 2013, "grade": "7.45 CGPA", "tier": "tier_4"}], "skills": [{"name": "SQL", "proficiency": "beginner", "endorsements": 12, "duration_months": 4}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 15, "duration_months": 27}, {"name": "Accounting", "proficiency": "intermediate", "endorsements": 15, "duration_months": 16}, {"name": "Rust", "proficiency": "beginner", "endorsements": 12, "duration_months": 15}, {"name": "Redux", "proficiency": "intermediate", "endorsements": 2, "duration_months": 19}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 14, "duration_months": 25}, {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 10, "duration_months": 21}, {"name": "REST APIs", "proficiency": "intermediate", "endorsements": 6, "duration_months": 18}, {"name": "Spark", "proficiency": "beginner", "endorsements": 4, "duration_months": 14}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 39.5, "signup_date": "2024-10-03", "last_active_date": "2026-01-31", "open_to_work_flag": false, "profile_views_received_30d": 52, "applications_submitted_30d": 6, "recruiter_response_rate": 0.46, "avg_response_time_hours": 268.3, "skill_assessment_scores": {}, "connection_count": 176, "endorsements_received": 35, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 7.6, "max": 8.2}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 95, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.8, "offer_acceptance_rate": 0.44, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000041", "profile": {"anonymized_name": "Anjali Khanna", "headline": "Operations Manager | Helping teams scale", "summary": "Professional with 13.7+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Delhi, Delhi", "country": "India", "years_of_experience": 13.7, "current_title": "Operations Manager", "current_company": "Hooli", "current_company_size": "1001-5000", "current_industry": "Software"}, "career_history": [{"company": "Hooli", "title": "Operations Manager", "start_date": "2022-12-14", "end_date": null, "duration_months": 42, "is_current": true, "industry": "Software", "company_size": "1001-5000", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Acme Corp", "title": "Business Analyst", "start_date": "2021-03-24", "end_date": "2022-12-14", "duration_months": 21, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Wayne Enterprises", "title": "Content Writer", "start_date": "2019-03-21", "end_date": "2021-03-10", "duration_months": 24, "is_current": false, "industry": "Conglomerate", "company_size": "10001+", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}, {"company": "Infosys", "title": "Content Writer", "start_date": "2014-12-05", "end_date": "2019-03-14", "duration_months": 52, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Dunder Mifflin", "title": "Mechanical Engineer", "start_date": "2013-01-07", "end_date": "2014-11-28", "duration_months": 23, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}], "education": [{"institution": "Local Engineering College", "degree": "M.S.", "field_of_study": "Machine Learning", "start_year": 2013, "end_year": 2018, "grade": "8.73 CGPA", "tier": "tier_4"}], "skills": [{"name": "Airflow", "proficiency": "intermediate", "endorsements": 13, "duration_months": 20}, {"name": "SQL", "proficiency": "intermediate", "endorsements": 12, "duration_months": 12}, {"name": "Go", "proficiency": "beginner", "endorsements": 13, "duration_months": 8}, {"name": "GCP", "proficiency": "beginner", "endorsements": 0, "duration_months": 3}, {"name": "Figma", "proficiency": "beginner", "endorsements": 14, "duration_months": 16}, {"name": "React", "proficiency": "intermediate", "endorsements": 15, "duration_months": 22}, {"name": "Webpack", "proficiency": "beginner", "endorsements": 3, "duration_months": 9}, {"name": "Kubernetes", "proficiency": "beginner", "endorsements": 5, "duration_months": 3}, {"name": "Angular", "proficiency": "intermediate", "endorsements": 2, "duration_months": 31}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 75.9, "signup_date": "2025-08-18", "last_active_date": "2026-03-16", "open_to_work_flag": true, "profile_views_received_30d": 3, "applications_submitted_30d": 9, "recruiter_response_rate": 0.07, "avg_response_time_hours": 135.3, "skill_assessment_scores": {}, "connection_count": 143, "endorsements_received": 19, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 5.4, "max": 9.3}, "preferred_work_mode": "remote", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 150, "saved_by_recruiters_30d": 6, "interview_completion_rate": 0.85, "offer_acceptance_rate": 0.16, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000042", "profile": {"anonymized_name": "Riya Mukherjee", "headline": "HR Manager | 5.0+ yrs experience", "summary": "Professional with 5.0+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Berlin", "country": "Germany", "years_of_experience": 5.0, "current_title": "HR Manager", "current_company": "Wayne Enterprises", "current_company_size": "10001+", "current_industry": "Conglomerate"}, "career_history": [{"company": "Wayne Enterprises", "title": "HR Manager", "start_date": "2024-04-07", "end_date": null, "duration_months": 26, "is_current": true, "industry": "Conglomerate", "company_size": "10001+", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}, {"company": "Wipro", "title": "Business Analyst", "start_date": "2023-03-14", "end_date": "2024-03-08", "duration_months": 12, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}, {"company": "Infosys", "title": "Customer Support", "start_date": "2021-04-23", "end_date": "2023-01-13", "duration_months": 21, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}], "education": [{"institution": "SRM Chennai", "degree": "B.Tech", "field_of_study": "Civil Engineering", "start_year": 2014, "end_year": 2019, "grade": "7.40 CGPA", "tier": "tier_3"}, {"institution": "Symbiosis International", "degree": "B.E.", "field_of_study": "Mathematics", "start_year": 2017, "end_year": 2021, "grade": "8.07 CGPA", "tier": "tier_3"}], "skills": [{"name": "Project Management", "proficiency": "intermediate", "endorsements": 3, "duration_months": 33}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 3, "duration_months": 10}, {"name": "Marketing", "proficiency": "intermediate", "endorsements": 10, "duration_months": 24}, {"name": "SAP", "proficiency": "beginner", "endorsements": 12, "duration_months": 18}, {"name": "Illustrator", "proficiency": "beginner", "endorsements": 11, "duration_months": 14}, {"name": "Node.js", "proficiency": "intermediate", "endorsements": 3, "duration_months": 14}, {"name": "YOLO", "proficiency": "intermediate", "endorsements": 5, "duration_months": 22}, {"name": "Tailwind", "proficiency": "beginner", "endorsements": 8, "duration_months": 16}, {"name": "CSS", "proficiency": "beginner", "endorsements": 11, "duration_months": 12}, {"name": "Figma", "proficiency": "beginner", "endorsements": 10, "duration_months": 7}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2022}, {"name": "Scrum Master Certified", "issuer": "Scrum Alliance", "year": 2021}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 58.6, "signup_date": "2025-05-02", "last_active_date": "2025-10-23", "open_to_work_flag": false, "profile_views_received_30d": 57, "applications_submitted_30d": 8, "recruiter_response_rate": 0.58, "avg_response_time_hours": 24.8, "skill_assessment_scores": {}, "connection_count": 591, "endorsements_received": 29, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 10.5, "max": 18.8}, "preferred_work_mode": "flexible", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 34, "saved_by_recruiters_30d": 9, "interview_completion_rate": 0.35, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000043", "profile": {"anonymized_name": "Aarav Sen", "headline": "Cloud Engineer | Full-stack development", "summary": "Software engineer with 8.3 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Chandigarh, Chandigarh", "country": "India", "years_of_experience": 8.3, "current_title": "Cloud Engineer", "current_company": "Swiggy", "current_company_size": "5001-10000", "current_industry": "Food Delivery"}, "career_history": [{"company": "Swiggy", "title": "Cloud Engineer", "start_date": "2023-12-09", "end_date": null, "duration_months": 30, "is_current": true, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "HCL", "title": ".NET Developer", "start_date": "2021-12-19", "end_date": "2023-12-09", "duration_months": 24, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Cloud infrastructure and DevOps work at an enterprise SaaS company. Owned the AWS account architecture (VPC, IAM, networking), the Terraform modules for our service deployments, and the Kubernetes cluster operations. Designed the CI/CD pipelines (GitLab CI + ArgoCD) and the monitoring stack (Prometheus, Grafana, Loki). Strong on the infra and ops side; haven't done much application development."}, {"company": "Mindtree", "title": "Frontend Engineer", "start_date": "2019-11-30", "end_date": "2021-12-19", "duration_months": 25, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Initech", "title": "DevOps Engineer", "start_date": "2018-04-09", "end_date": "2019-11-30", "duration_months": 20, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}], "education": [{"institution": "Regional Technical Institute", "degree": "M.E.", "field_of_study": "Electronics", "start_year": 2012, "end_year": 2017, "grade": "8.77 CGPA", "tier": "tier_4"}, {"institution": "Regional Technical Institute", "degree": "B.Sc", "field_of_study": "Physics", "start_year": 2001, "end_year": 2005, "grade": "9.29 CGPA", "tier": "tier_4"}], "skills": [{"name": "Elasticsearch", "proficiency": "advanced", "endorsements": 54, "duration_months": 44}, {"name": "OpenSearch", "proficiency": "intermediate", "endorsements": 21, "duration_months": 34}, {"name": "Airflow", "proficiency": "beginner", "endorsements": 5, "duration_months": 2}, {"name": "Kubeflow", "proficiency": "advanced", "endorsements": 12, "duration_months": 55}, {"name": "Fine-tuning LLMs", "proficiency": "intermediate", "endorsements": 8, "duration_months": 14}, {"name": "Haystack", "proficiency": "advanced", "endorsements": 11, "duration_months": 27}, {"name": "OpenCV", "proficiency": "advanced", "endorsements": 54, "duration_months": 33}, {"name": "TensorFlow", "proficiency": "intermediate", "endorsements": 0, "duration_months": 19}, {"name": "PEFT", "proficiency": "intermediate", "endorsements": 5, "duration_months": 21}, {"name": "LangChain", "proficiency": "intermediate", "endorsements": 37, "duration_months": 25}, {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 4, "duration_months": 26}, {"name": "Reinforcement Learning", "proficiency": "advanced", "endorsements": 0, "duration_months": 30}, {"name": "Deep Learning", "proficiency": "intermediate", "endorsements": 0, "duration_months": 16}, {"name": "Image Classification", "proficiency": "advanced", "endorsements": 24, "duration_months": 50}, {"name": "Node.js", "proficiency": "intermediate", "endorsements": 12, "duration_months": 24}, {"name": "Project Management", "proficiency": "intermediate", "endorsements": 0, "duration_months": 36}, {"name": "Feature Engineering", "proficiency": "intermediate", "endorsements": 0, "duration_months": 20}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 57.0, "signup_date": "2024-09-30", "last_active_date": "2026-01-01", "open_to_work_flag": false, "profile_views_received_30d": 38, "applications_submitted_30d": 8, "recruiter_response_rate": 0.04, "avg_response_time_hours": 223.5, "skill_assessment_scores": {}, "connection_count": 102, "endorsements_received": 39, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 6.3, "max": 21.2}, "preferred_work_mode": "remote", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 167, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.72, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000044", "profile": {"anonymized_name": "Vihaan Naidu", "headline": "Frontend Engineer | Backend systems & APIs", "summary": "Software engineer with 5.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've spent most of my career on web and API development \u2014 Python/Django and Node.js mostly. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Indore, Madhya Pradesh", "country": "India", "years_of_experience": 5.7, "current_title": "Frontend Engineer", "current_company": "Tech Mahindra", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Tech Mahindra", "title": "Frontend Engineer", "start_date": "2022-04-18", "end_date": null, "duration_months": 50, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}, {"company": "Hooli", "title": "DevOps Engineer", "start_date": "2020-10-25", "end_date": "2022-04-18", "duration_months": 18, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."}], "education": [{"institution": "Chandigarh University", "degree": "M.Sc", "field_of_study": "Information Technology", "start_year": 2016, "end_year": 2021, "grade": "84%", "tier": "tier_3"}, {"institution": "Amity University", "degree": "B.E.", "field_of_study": "Civil Engineering", "start_year": 2002, "end_year": 2006, "grade": "9.33 CGPA", "tier": "tier_3"}], "skills": [{"name": "Hadoop", "proficiency": "beginner", "endorsements": 3, "duration_months": 3}, {"name": "JavaScript", "proficiency": "beginner", "endorsements": 5, "duration_months": 18}, {"name": "Databricks", "proficiency": "beginner", "endorsements": 13, "duration_months": 5}, {"name": "Python", "proficiency": "intermediate", "endorsements": 2, "duration_months": 26}, {"name": "dbt", "proficiency": "intermediate", "endorsements": 3, "duration_months": 16}, {"name": "CI/CD", "proficiency": "beginner", "endorsements": 11, "duration_months": 13}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 30.6, "signup_date": "2023-02-16", "last_active_date": "2025-12-11", "open_to_work_flag": false, "profile_views_received_30d": 78, "applications_submitted_30d": 7, "recruiter_response_rate": 0.66, "avg_response_time_hours": 179.1, "skill_assessment_scores": {}, "connection_count": 58, "endorsements_received": 38, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 8.8, "max": 12.3}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": 6.5, "search_appearance_30d": 288, "saved_by_recruiters_30d": 18, "interview_completion_rate": 0.66, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000045", "profile": {"anonymized_name": "Vikram Mittal", "headline": "Project Manager | 12.2+ yrs experience", "summary": "Professional with 12.2+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Indore, Madhya Pradesh", "country": "India", "years_of_experience": 12.2, "current_title": "Project Manager", "current_company": "Initech", "current_company_size": "51-200", "current_industry": "Software"}, "career_history": [{"company": "Initech", "title": "Project Manager", "start_date": "2025-03-03", "end_date": null, "duration_months": 15, "is_current": true, "industry": "Software", "company_size": "51-200", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}, {"company": "Pied Piper", "title": "Civil Engineer", "start_date": "2024-02-07", "end_date": "2025-03-03", "duration_months": 13, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Infosys", "title": "Marketing Manager", "start_date": "2023-01-29", "end_date": "2024-01-24", "duration_months": 12, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Wayne Enterprises", "title": "Content Writer", "start_date": "2021-10-06", "end_date": "2023-01-29", "duration_months": 16, "is_current": false, "industry": "Conglomerate", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "TCS", "title": "Graphic Designer", "start_date": "2018-08-09", "end_date": "2021-09-22", "duration_months": 38, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Wayne Enterprises", "title": "Graphic Designer", "start_date": "2016-04-21", "end_date": "2018-08-09", "duration_months": 28, "is_current": false, "industry": "Conglomerate", "company_size": "10001+", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}, {"company": "Stark Industries", "title": "Operations Manager", "start_date": "2014-07-17", "end_date": "2016-04-07", "duration_months": 21, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}], "education": [{"institution": "Generic State University", "degree": "M.E.", "field_of_study": "Statistics", "start_year": 2002, "end_year": 2005, "grade": "7.91 CGPA", "tier": "tier_4"}, {"institution": "KIIT University", "degree": "B.Tech", "field_of_study": "Machine Learning", "start_year": 2016, "end_year": 2020, "grade": "86%", "tier": "tier_3"}], "skills": [{"name": "GCP", "proficiency": "beginner", "endorsements": 1, "duration_months": 3}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 2, "duration_months": 12}, {"name": "Redux", "proficiency": "intermediate", "endorsements": 14, "duration_months": 25}, {"name": "PostgreSQL", "proficiency": "intermediate", "endorsements": 1, "duration_months": 36}, {"name": "Airflow", "proficiency": "intermediate", "endorsements": 5, "duration_months": 18}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 13, "duration_months": 27}], "certifications": [{"name": "Scrum Master Certified", "issuer": "Scrum Alliance", "year": 2024}, {"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2024}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 25.4, "signup_date": "2023-04-15", "last_active_date": "2026-04-24", "open_to_work_flag": true, "profile_views_received_30d": 37, "applications_submitted_30d": 0, "recruiter_response_rate": 0.62, "avg_response_time_hours": 20.8, "skill_assessment_scores": {}, "connection_count": 490, "endorsements_received": 2, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 9.7, "max": 12.3}, "preferred_work_mode": "hybrid", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 23, "saved_by_recruiters_30d": 6, "interview_completion_rate": 0.33, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000046", "profile": {"anonymized_name": "Dev Nair", "headline": "Mechanical Engineer | 7.8+ yrs experience", "summary": "Professional with 7.8+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "London", "country": "UK", "years_of_experience": 7.8, "current_title": "Mechanical Engineer", "current_company": "Hooli", "current_company_size": "1001-5000", "current_industry": "Software"}, "career_history": [{"company": "Hooli", "title": "Mechanical Engineer", "start_date": "2023-12-09", "end_date": null, "duration_months": 30, "is_current": true, "industry": "Software", "company_size": "1001-5000", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Pied Piper", "title": "HR Manager", "start_date": "2021-02-22", "end_date": "2023-11-09", "duration_months": 33, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Hooli", "title": "Marketing Manager", "start_date": "2018-08-23", "end_date": "2021-02-08", "duration_months": 30, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}], "education": [{"institution": "Christ University", "degree": "M.Tech", "field_of_study": "Electrical Engineering", "start_year": 2014, "end_year": 2017, "grade": "9.49 CGPA", "tier": "tier_3"}, {"institution": "Chandigarh University", "degree": "B.E.", "field_of_study": "Statistics", "start_year": 2004, "end_year": 2009, "grade": "7.94 CGPA", "tier": "tier_3"}], "skills": [{"name": "Agile", "proficiency": "beginner", "endorsements": 1, "duration_months": 9}, {"name": "Scrum", "proficiency": "intermediate", "endorsements": 11, "duration_months": 28}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 6, "duration_months": 35}, {"name": "React", "proficiency": "beginner", "endorsements": 7, "duration_months": 7}, {"name": "Azure", "proficiency": "intermediate", "endorsements": 3, "duration_months": 36}, {"name": "ETL", "proficiency": "beginner", "endorsements": 2, "duration_months": 8}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 74.8, "signup_date": "2023-02-12", "last_active_date": "2026-02-20", "open_to_work_flag": false, "profile_views_received_30d": 80, "applications_submitted_30d": 6, "recruiter_response_rate": 0.41, "avg_response_time_hours": 209.8, "skill_assessment_scores": {}, "connection_count": 301, "endorsements_received": 29, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 7.6, "max": 28.0}, "preferred_work_mode": "remote", "willing_to_relocate": true, "github_activity_score": 20.9, "search_appearance_30d": 166, "saved_by_recruiters_30d": 7, "interview_completion_rate": 0.4, "offer_acceptance_rate": 0.34, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000047", "profile": {"anonymized_name": "Avni Bansal", "headline": "Project Manager | Helping teams scale", "summary": "Professional with 2.4+ years of experience. My professional background is in marketing manager \u2014 I've built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Kochi, Kerala", "country": "India", "years_of_experience": 2.4, "current_title": "Project Manager", "current_company": "TCS", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "TCS", "title": "Project Manager", "start_date": "2024-02-07", "end_date": null, "duration_months": 28, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}], "education": [{"institution": "Amity University", "degree": "B.Sc", "field_of_study": "Mechanical Engineering", "start_year": 2011, "end_year": 2016, "grade": "7.18 CGPA", "tier": "tier_3"}], "skills": [{"name": "FastAPI", "proficiency": "beginner", "endorsements": 8, "duration_months": 4}, {"name": "Java", "proficiency": "intermediate", "endorsements": 5, "duration_months": 28}, {"name": "Excel", "proficiency": "intermediate", "endorsements": 10, "duration_months": 9}, {"name": "Tally", "proficiency": "intermediate", "endorsements": 10, "duration_months": 9}, {"name": "SQL", "proficiency": "intermediate", "endorsements": 11, "duration_months": 15}, {"name": "Scrum", "proficiency": "intermediate", "endorsements": 1, "duration_months": 13}, {"name": "Hadoop", "proficiency": "beginner", "endorsements": 10, "duration_months": 17}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 79.7, "signup_date": "2025-06-07", "last_active_date": "2026-03-22", "open_to_work_flag": false, "profile_views_received_30d": 34, "applications_submitted_30d": 5, "recruiter_response_rate": 0.39, "avg_response_time_hours": 109.1, "skill_assessment_scores": {}, "connection_count": 444, "endorsements_received": 48, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 13.3, "max": 19.2}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 14, "saved_by_recruiters_30d": 4, "interview_completion_rate": 0.54, "offer_acceptance_rate": 0.21, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000048", "profile": {"anonymized_name": "Vihaan Saxena", "headline": "Mobile Developer | Full-stack development", "summary": "Software engineer with 9.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I've been keeping up with AI/ML at a self-learner level \u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \u2014 but I haven't done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Hyderabad, Telangana", "country": "India", "years_of_experience": 9.7, "current_title": "Mobile Developer", "current_company": "CRED", "current_company_size": "1001-5000", "current_industry": "Fintech"}, "career_history": [{"company": "CRED", "title": "Mobile Developer", "start_date": "2024-02-07", "end_date": null, "duration_months": 28, "is_current": true, "industry": "Fintech", "company_size": "1001-5000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company's design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \u2014 accessibility, performance, animations \u2014 but limited backend exposure."}, {"company": "Cognizant", "title": "Full Stack Developer", "start_date": "2022-06-17", "end_date": "2024-02-07", "duration_months": 20, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Stark Industries", "title": "Mobile Developer", "start_date": "2018-02-08", "end_date": "2022-04-18", "duration_months": 51, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Java backend development at a large enterprise \u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}, {"company": "Initech", "title": "Full Stack Developer", "start_date": "2016-11-15", "end_date": "2018-02-08", "duration_months": 15, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}], "education": [{"institution": "VIT Chennai", "degree": "B.E.", "field_of_study": "Data Science", "start_year": 2005, "end_year": 2009, "grade": "84%", "tier": "tier_3"}, {"institution": "Anna University", "degree": "B.Tech", "field_of_study": "Statistics", "start_year": 2011, "end_year": 2016, "grade": "6.96 CGPA", "tier": "tier_2"}], "skills": [{"name": "Hadoop", "proficiency": "beginner", "endorsements": 2, "duration_months": 15}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 13, "duration_months": 26}, {"name": "Vue.js", "proficiency": "intermediate", "endorsements": 2, "duration_months": 24}, {"name": "Content Writing", "proficiency": "intermediate", "endorsements": 11, "duration_months": 26}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 1, "duration_months": 31}], "certifications": [{"name": "Scrum Master Certified", "issuer": "Scrum Alliance", "year": 2023}, {"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2024}], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 62.2, "signup_date": "2026-03-28", "last_active_date": "2026-04-06", "open_to_work_flag": true, "profile_views_received_30d": 58, "applications_submitted_30d": 3, "recruiter_response_rate": 0.65, "avg_response_time_hours": 97.9, "skill_assessment_scores": {}, "connection_count": 225, "endorsements_received": 48, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 12.6, "max": 26.6}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 131, "saved_by_recruiters_30d": 5, "interview_completion_rate": 0.42, "offer_acceptance_rate": 0.4, "verified_email": false, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000049", "profile": {"anonymized_name": "Tanya Chowdary", "headline": "Mechanical Engineer | Helping teams scale", "summary": "Professional with 11.8+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Berlin", "country": "Germany", "years_of_experience": 11.8, "current_title": "Mechanical Engineer", "current_company": "Wayne Enterprises", "current_company_size": "10001+", "current_industry": "Conglomerate"}, "career_history": [{"company": "Wayne Enterprises", "title": "Mechanical Engineer", "start_date": "2022-05-18", "end_date": null, "duration_months": 49, "is_current": true, "industry": "Conglomerate", "company_size": "10001+", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}, {"company": "Hooli", "title": "Business Analyst", "start_date": "2017-11-10", "end_date": "2022-04-18", "duration_months": 54, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Wayne Enterprises", "title": "Sales Executive", "start_date": "2014-09-27", "end_date": "2017-11-10", "duration_months": 38, "is_current": false, "industry": "Conglomerate", "company_size": "10001+", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}], "education": [{"institution": "Symbiosis International", "degree": "M.Tech", "field_of_study": "Mathematics", "start_year": 2016, "end_year": 2019, "grade": "7.32 CGPA", "tier": "tier_3"}], "skills": [{"name": "TypeScript", "proficiency": "beginner", "endorsements": 0, "duration_months": 8}, {"name": "Rust", "proficiency": "intermediate", "endorsements": 1, "duration_months": 16}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 6, "duration_months": 11}, {"name": "Apache Beam", "proficiency": "beginner", "endorsements": 10, "duration_months": 4}, {"name": "GraphQL", "proficiency": "intermediate", "endorsements": 14, "duration_months": 16}, {"name": "Kubernetes", "proficiency": "intermediate", "endorsements": 5, "duration_months": 13}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 64.0, "signup_date": "2025-09-12", "last_active_date": "2026-05-13", "open_to_work_flag": false, "profile_views_received_30d": 6, "applications_submitted_30d": 2, "recruiter_response_rate": 0.59, "avg_response_time_hours": 109.3, "skill_assessment_scores": {}, "connection_count": 514, "endorsements_received": 12, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 12.9, "max": 18.8}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 100, "saved_by_recruiters_30d": 6, "interview_completion_rate": 0.56, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0000050", "profile": {"anonymized_name": "Naina Bose", "headline": "Business Analyst | Helping teams scale", "summary": "Professional with 13.5+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work \u2014 I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Gurgaon, Haryana", "country": "India", "years_of_experience": 13.5, "current_title": "Business Analyst", "current_company": "Infosys", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Infosys", "title": "Business Analyst", "start_date": "2022-09-15", "end_date": null, "duration_months": 45, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "TCS", "title": "Marketing Manager", "start_date": "2019-08-02", "end_date": "2022-07-17", "duration_months": 36, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}, {"company": "Hooli", "title": "HR Manager", "start_date": "2017-03-15", "end_date": "2019-07-03", "duration_months": 28, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Acme Corp", "title": "Content Writer", "start_date": "2012-12-22", "end_date": "2017-03-01", "duration_months": 51, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Senior accounting role at a mid-sized company \u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}], "education": [{"institution": "VIT Chennai", "degree": "Ph.D", "field_of_study": "Artificial Intelligence", "start_year": 2001, "end_year": 2005, "grade": "6.66 CGPA", "tier": "tier_3"}, {"institution": "Georgia Tech", "degree": "B.Sc", "field_of_study": "Artificial Intelligence", "start_year": 2013, "end_year": 2017, "grade": "7.81 CGPA", "tier": "tier_1"}], "skills": [{"name": "gRPC", "proficiency": "intermediate", "endorsements": 9, "duration_months": 11}, {"name": "SEO", "proficiency": "beginner", "endorsements": 8, "duration_months": 18}, {"name": "Feature Engineering", "proficiency": "advanced", "endorsements": 4, "duration_months": 42}, {"name": "Marketing", "proficiency": "beginner", "endorsements": 0, "duration_months": 15}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 0, "duration_months": 15}, {"name": "Kafka", "proficiency": "beginner", "endorsements": 7, "duration_months": 13}, {"name": "Excel", "proficiency": "intermediate", "endorsements": 10, "duration_months": 23}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 42.5, "signup_date": "2023-01-23", "last_active_date": "2025-10-22", "open_to_work_flag": false, "profile_views_received_30d": 34, "applications_submitted_30d": 2, "recruiter_response_rate": 0.42, "avg_response_time_hours": 108.7, "skill_assessment_scores": {"Feature Engineering": 60.8}, "connection_count": 245, "endorsements_received": 22, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 7.6, "max": 22.9}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": 44.7, "search_appearance_30d": 87, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.58, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}]''')
print("loaded", len(sample_candidates), "candidates")


loaded 50 candidates


In [4]:
# Step 4: embed the JD and the candidates
# Uses sentence-transformers if it can reach the internet (Colab has internet).
# Falls back automatically to the scikit-learn TF-IDF engine if the model
# cannot be downloaded, so this cell never blocks the reproducibility check.

import numpy as np

texts = [candidate_text(c) for c in sample_candidates]

try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    jd_emb = model.encode([JD_TEXT], normalize_embeddings=True)[0]
    cand_emb = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
    engine_used = "sbert"
except Exception as e:
    print("sentence-transformers unavailable, falling back to tfidf:", e)
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import normalize as sk_normalize

    vectorizer = TfidfVectorizer(max_features=20000, stop_words="english", ngram_range=(1, 2))
    all_texts = [JD_TEXT] + texts
    tfidf = vectorizer.fit_transform(all_texts)
    n_comp = min(128, tfidf.shape[0] - 1, tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    reduced = sk_normalize(svd.fit_transform(tfidf))
    jd_emb = reduced[0].astype(np.float32)
    cand_emb = reduced[1:].astype(np.float32)
    engine_used = "tfidf"

print("embedding engine used:", engine_used)
print("candidate embedding shape:", cand_emb.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding engine used: sbert
candidate embedding shape: (50, 384)


In [5]:
# Step 5: score and rank
# This is the timed step. On the full 100k candidate file this must finish
# in under 5 minutes on CPU only with no network. On this 50 candidate
# sample it finishes in under a second.

import time

t0 = time.time()

sim_scores = cand_emb @ jd_emb

scored = []
for i, c in enumerate(sample_candidates):
    honeypot = is_honeypot(c)
    penalty = hard_penalty_gate(c)
    exp_fit = years_experience_fit(c.get("profile", {}).get("years_of_experience"))
    loc_fit = location_fit(c.get("profile", {}).get("location"), c.get("profile", {}).get("country"))
    behav = behavioral_multiplier(c)
    retrieval_hits = has_retrieval_evidence(c)

    score = sim_scores[i] * penalty * exp_fit * loc_fit * behav
    if honeypot:
        score = -1.0

    scored.append({
        "candidate_id": c["candidate_id"],
        "raw_score": float(score),
        "reasoning": build_reasoning(c, sim_scores[i], penalty, behav, retrieval_hits),
    })

elapsed = time.time() - t0
print(f"scored {len(scored)} candidates in {elapsed:.2f} seconds")


scored 50 candidates in 0.02 seconds


In [6]:
# Step 6: build the ranked output
# Same tie break rule as the real submission: score descending,
# candidate_id ascending on ties.

scored_sorted = sorted(scored, key=lambda r: (-r["raw_score"], r["candidate_id"]))

top_n = min(20, len(scored_sorted))
top = scored_sorted[:top_n]

max_s = max(r["raw_score"] for r in top)
min_s = min(r["raw_score"] for r in top)
span = max(max_s - min_s, 1e-9)

rows = []
for rank, r in enumerate(top, start=1):
    norm_score = 0.5 + 0.5 * (r["raw_score"] - min_s) / span
    rows.append((r["candidate_id"], rank, round(norm_score, 4), r["reasoning"]))

print(f"ranked top {top_n} candidates from this sample")


ranked top 20 candidates from this sample


In [7]:
# Step 7: write and show the output CSV
import csv

with open("sandbox_output.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["candidate_id", "rank", "score", "reasoning"])
    for row in rows:
        writer.writerow(row)

import pandas as pd
df = pd.read_csv("sandbox_output.csv")

,candidate_id,rank,score,reasoning
0,CAND_0000014,1,1.0000,"Frontend Engineer with 8.4 yrs, currently at Z..."
1,CAND_0000031,2,0.9316,"Recommendation Systems Engineer with 6.0 yrs, ..."
2,CAND_0000001,3,0.8625,"Backend Engineer with 6.9 yrs, currently at Mi..."
3,CAND_0000048,4,0.8530,"Mobile Developer with 9.7 yrs, currently at CR..."
4,CAND_0000023,5,0.7405,"Software Engineer with 3.7 yrs, currently at A..."
5,CAND_0000015,6,0.7102,"Software Engineer with 5.4 yrs, currently at R..."
6,CAND_0000010,7,0.7036,"Data Engineer with 4.6 yrs, currently at Ola (..."
7,CAND_0000007,8,0.7024,"Civil Engineer with 5.5 yrs, currently at Wipr..."
8,CAND_0000045,9,0.6752,"Project Manager with 12.2 yrs, currently at In..."
9,CAND_0000038,10,0.6398,"Java Developer with 6.7 yrs, currently at Swig..."


## What this notebook proves

It shows the same rank.py logic from the GitHub repo running end to end on a
small candidate sample, producing a valid ranked CSV, without needing the
full 100k candidate file, without any API key, and without a GPU for the
ranking step.

The full run on all 100k candidates is done locally with rank.py and
precompute_embeddings.py from the GitHub repo, which is what produces the
actual submission.csv for the hackathon.
